In [1]:
# CELL 1: CONFIG + HELPERS

import os, re, sys, gc, json, math, time, random, shutil, zipfile, hashlib, contextlib
from dataclasses import dataclass, asdict, field

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint


SMOKE_TEST = False

RUN_TAG = "p5_lsi_m4_kd_lora_v1" + ("_smoke" if SMOKE_TEST else "")
TEACHER_P1_TAG = "decoder_only_rope_gqa_swiglu_mtp4_v1"            # MODEL_TAG of the Model-3 pretraining notebook
STUDENT_P1_TAG = "decoder_only_msm_rope_gqa_swiglu_mtp3_v2"        # MODEL_TAG of the Model-4 pretraining notebook
TEACHER_RUN_TAG = "p5_lsi_m3_lora_v1_allchunks"                    # RUN_TAG of the Phase-5 Model-3 notebook (the fine-tuned teacher)

IS_KAGGLE = os.path.isdir("/kaggle/working")
INPUT_ROOT = "/kaggle/input"
WORK_DIR = ("/kaggle/working" if IS_KAGGLE else ".") + f"/{RUN_TAG}"
CKPT_DIR = f"{WORK_DIR}/checkpoints"
CACHE_DIR = f"{WORK_DIR}/cache"
OUT_DIR = f"{WORK_DIR}/submission"
for _d in (CKPT_DIR, CACHE_DIR, OUT_DIR):
    os.makedirs(_d, exist_ok=True)

# Optional manual overrides. Leave "" to auto-search /kaggle/input.
TOKENIZER_DIR = ""          # restrict the tokenizer search to this folder
TEACHER_P1_CKPT = ""        # full path of the Model-3 Phase-1 best.pt
STUDENT_P1_CKPT = ""        # full path of the Model-4 Phase-1 best.pt
TEACHER_LSI_CKPT = ""


TOKENIZER_KIND = "custom"
TOKENIZER_CHECK = True            # run the pretrained-LM loss check with the teacher AND the student
TOKENIZER_STRICT = True           # raise if the tokenizer is far off the pretraining val_loss of either model
LM_CHECK_DOCS = 16
LM_CHECK_MARGIN = 3.0

# ---- data ----
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
PHASE5_FILES = {
    "train": "Phase_5_LSI/Train/train.json",
    "dev":   "Phase_5_LSI/Train/dev.json",
    "test":  "Phase_5_LSI/Test/test.json",
}
SIMPLIFY_PLACEHOLDERS = True      # "<ENTITY>" -> "ENTITY" before tokenising (anonymised corpus)
MAX_DOC_TOKENS = 40_000           # pure safety cap (~52 chunks, far above the p99 doc): head 70% + tail 30% are kept.
KEEP_HEAD_FRAC = 0.7


CHUNK_LEN = 1024                  # = the context length both models were pretrained with
STRIDE = 768                      # 256-token overlap between consecutive chunks
OVERLAP = CHUNK_LEN - STRIDE
ENC_MB_TRAIN = 24
ENC_MB_EVAL = 64


KD_ALPHA = 0.5                    # 0 -> no label-level distillation
KD_TEMP = 2.0
ATT_KD_W = 0.3                    # 0 -> off
CHUNK_KD_W = 0.3                  # 0 -> off
CHUNK_KD_TEMP = 1.0
LOGIT_MSE_W = 0.0                 # optional Ba&Caruana logit matching (0 = off)

USE_TEACHER = (KD_ALPHA > 0) or (ATT_KD_W > 0) or (CHUNK_KD_W > 0) or (LOGIT_MSE_W > 0)


TEACHER_LORA_ALPHA = 64.0         # LORA_ALPHA of the Model-3 Phase-5 run
TEACHER_DOC_HEADS = 8
TEACHER_HEAD_DROPOUT = 0.1
TEACHER_LORA_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "w_gate", "w_up", "w_down")
TEACHER_CACHE_SAVE_SEC = 300      # teacher cache is flushed to disk at most this often (resumable)
TEACHER_MAX_H = 3.0

TEACHER_STRICT = True
TEACHER_CHECK_TOL = 0.03
PATH_KD_TARGET = "fused"

# ---- student LoRA (Model 4) ----
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
USE_RSLORA = False

STUDENT_LORA_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "w_gate", "w_up", "w_down",
                        "wq_proj", "wk_proj", "wv_proj", "write_out", "gate_proj")
STUDENT_TRAIN_MEM_INIT = True     # the learned latent memory (32 x d_model) and final_norm are trained as well
FT_DROPOUT = 0.1
GRAD_CKPT = True                  # gradient checkpointing of the student's blocks (less memory, ~30% slower)

# ---- hierarchical document head (student) ----
HEAD_HIDDEN = 512
HEAD_DROPOUT = 0.1
DOC_LAYERS = 1
DOC_HEADS = 8
DOC_FF = 1024
MAX_POS_EMB = 64                  # learned chunk-index embeddings (needs n_chunks <= 64)
REL_BUCKETS = 8
AUX_W = 0.3                       # weight of the per-path auxiliary losses
POS_WEIGHT_CAP = 5.0              # pos_weight = clip(sqrt(neg/pos), 1, cap)

# ---- optimisation ----
EPOCHS = 5
TRAIN_TOKEN_BUDGET = 24 * CHUNK_LEN
EVAL_TOKEN_BUDGET = 64 * CHUNK_LEN
MAX_DOCS_PER_BATCH = 48
DOC_NORM = 16.0                       # loss = sum over docs / DOC_NORM  (every doc weighs the same)
LORA_LR = 4e-4
HEAD_LR = 1e-3
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 200
MIN_LR_RATIO = 0.05
GRAD_CLIP = 1.0
USE_EMA = True
EMA_DECAY = 0.998
SAVE_EVERY = 300
LOG_EVERY = 50
SEED = 1234

# ---- time budget (free tier!) ----
TIME_BUDGET_H = 12.0                   # wall-clock limit of THIS session
FINAL_RESERVE_H = 1.0                 # kept free for dev tuning + test prediction
TRAIN_HOURS = 11.0

if SMOKE_TEST:
    N_SMOKE = {"train": 600, "dev": 300, "test": 200}
    EPOCHS, SAVE_EVERY, LOG_EVERY, WARMUP_STEPS = 1, 10, 5, 10
else:
    N_SMOKE = None


# small helpers

def log(*a):
    print(*a, flush=True)


def fmt_time(sec):
    sec = int(max(sec, 0))
    return f"{sec // 3600}h{(sec % 3600) // 60:02d}m"


def read_json(path):
    try:
        with open(path) as f:
            return json.load(f)
    except Exception:
        return None


def atomic_json(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=1)
    os.replace(tmp, path)


def atomic_torch_save(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)


def atomic_npy(arr, path):
    tmp = path + ".tmp"
    with open(tmp, "wb") as f:
        np.save(f, arr)
    os.replace(tmp, path)


def walk_find(filename, roots):
    hits = []
    for root in roots:
        if not os.path.isdir(root):
            continue
        for d, _, files in os.walk(root):
            if filename in files:
                hits.append(d)
    return hits


def get_hf_token():
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"]
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF")      # Kaggle secret named HF
    except Exception:
        return None


T_START = time.time()          # session clock: setup + teacher stage count against TIME_BUDGET_H

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = {"enabled": False, "dtype": torch.float16}
if DEVICE.type == "cuda":
    AMP["enabled"] = True
    AMP["dtype"] = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
    torch.backends.cuda.matmul.allow_tf32 = True
    log(f"GPU: {torch.cuda.get_device_name()} | autocast dtype: {AMP['dtype']}")
else:
    log("WARNING: no GPU visible - this will be extremely slow")


def amp_ctx():
    """mixed precision for the backbones"""
    return torch.autocast(device_type="cuda", dtype=AMP["dtype"], enabled=AMP["enabled"])


def no_amp():
    """full fp32 for the (small) hierarchical heads: softmax / logsumexp / KL terms stay exact"""
    if AMP["enabled"]:
        return torch.autocast(device_type="cuda", enabled=False)
    return contextlib.nullcontext()


def make_grad_scaler(enabled):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):                       # older torch
        return torch.cuda.amp.GradScaler(enabled=enabled)


# locate checkpoints in /kaggle/input

def find_phase1_ckpt(tag, override=""):
    
    if override:
        return override
    cands, seen = [], []
    for fname, mname in (("best.pt", "best_meta.json"), ("latest.pt", "latest_meta.json")):
        for d in walk_find(fname, [INPUT_ROOT]):
            meta = read_json(os.path.join(d, mname)) or {}
            seen.append((os.path.join(d, fname), meta.get("tag")))
            if meta.get("tag") == tag:
                if fname == "best.pt":
                    cands.append((0, float(meta.get("val_loss", 1e9)), os.path.join(d, fname)))
                else:
                    cands.append((1, -float(meta.get("global_step", 0)), os.path.join(d, fname)))
    if not cands:
        raise FileNotFoundError(
            f"No Phase-1 checkpoint with tag '{tag}' under {INPUT_ROOT}. Found: {seen or 'no best.pt/latest.pt at all'}. "
            "Attach that model's pretraining output as an input, or set the matching *_P1_CKPT override (full path of best.pt).")
    cands.sort()
    return cands[0][2]


def find_teacher_lsi():
    
    if TEACHER_LSI_CKPT:
        d = os.path.dirname(TEACHER_LSI_CKPT)
        meta = read_json(os.path.join(d, "lsi_best_meta.json"))
        if meta is None:
            raise FileNotFoundError(f"lsi_best_meta.json not found next to {TEACHER_LSI_CKPT}")
        return d, meta
    best, seen = None, []
    for d in walk_find("lsi_best_meta.json", [INPUT_ROOT]):
        meta = read_json(os.path.join(d, "lsi_best_meta.json"))
        seen.append((d, (meta or {}).get("tag")))
        if not meta or meta.get("tag") != TEACHER_RUN_TAG or not os.path.exists(os.path.join(d, "lsi_best.pt")):
            continue
        if best is None or meta.get("dev_macro_f1", -1) > best[1].get("dev_macro_f1", -1):
            best = (d, meta)
    if best is None:
        raise FileNotFoundError(
            f"Fine-tuned teacher not found: need the output of the Phase-5 Model-3 notebook (lsi_best.pt + lsi_best_meta.json, tag "
            f"'{TEACHER_RUN_TAG}') under {INPUT_ROOT}. Found: {seen or 'no lsi_best_meta.json at all'}. Attach that notebook's "
            "output as an input (or set TEACHER_LSI_CKPT), or set KD_ALPHA = ATT_KD_W = CHUNK_KD_W = 0 for a no-teacher baseline.")
    return best


assert HEAD_HIDDEN % DOC_HEADS == 0
log("config ready | run tag:", RUN_TAG, "| smoke:", SMOKE_TEST, "| teacher stage:", "ON" if USE_TEACHER else "OFF (baseline run)")


GPU: Tesla T4 | autocast dtype: torch.float16
config ready | run tag: p5_lsi_m4_kd_lora_v1 | smoke: False | teacher stage: ON


In [2]:
# CELL 2: MODEL DEFINITIONS

@dataclass
class M3_ModelConfig:
    vocab_size: int = 50048                       # embedding rows (padded)
    real_vocab: int = 50000
    d_model: int = 640
    n_layers: int = 8
    n_heads: int = 10
    n_kv_heads: int = 2
    ffn_hidden: int = 1536
    dropout: float = 0.1
    max_len: int = 1088
    mtp_heads: int = 3
    mtp_weights: list = field(default_factory=lambda: [1.0, 0.5, 0.25])
    pad_id: int = 0
    unk_id: int = 1
    eos_id: int = 2
    bos_id: int = 3
    mask_id: int = 4


@dataclass
class M4_ModelConfig:
    vocab_size: int = 50000
    d_model: int = 512
    n_heads: int = 8
    n_kv_heads: int = 2
    ffn_hidden: int = 1792
    attn_dropout: float = 0.0
    ffn_dropout: float = 0.10
    layer_types: tuple = ("local_small", "local_large", "memory") * 3
    window_small: int = 128
    window_large: int = 256
    n_mem_chunks: int = 8
    chunk_size: int = 128
    mem_per_chunk: int = 4
    max_len: int = 1088
    pad_id: int = 0
    unk_id: int = 1
    eos_id: int = 2
    n_mtp: int = 3
    mtp_weights: tuple = (1.0, 0.5, 0.25)
    mtp_pos_frac: float = 0.25

    @property
    def n_layers(self):
        return len(self.layer_types)


class M3_RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        variance = x.float().pow(2).mean(-1, keepdim=True)
        inv_rms = torch.rsqrt(variance + self.eps).to(x.dtype)
        return x * inv_rms * self.weight.to(x.dtype)


def M3_build_rope(head_dim, max_len, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    freqs = torch.outer(torch.arange(max_len).float(), inv_freq)
    return freqs.cos(), freqs.sin()          # each (max_len, head_dim/2)


def M3_apply_rope(x, cos, sin):
    cos = cos.to(dtype=x.dtype)
    sin = sin.to(dtype=x.dtype)

    x1, x2 = x.chunk(2, dim=-1)

    return torch.cat([
        x1 * cos - x2 * sin,
        x1 * sin + x2 * cos
    ], dim=-1)


def M3_repeat_kv(x, groups):
    # (B, KV, T, hd) -> (B, KV*groups, T, hd); query head h reads kv head h // groups
    if groups == 1:
        return x
    B, KV, T, hd = x.shape
    return x[:, :, None].expand(B, KV, groups, T, hd).reshape(B, KV * groups, T, hd)


class M3_GQAttention(nn.Module):

    """causal grouped-query self-attention with RoPE (10 query heads share 2 key/value heads)"""

    def __init__(self, cfg):
        super().__init__()
        self.h = cfg.n_heads
        self.kv = cfg.n_kv_heads
        self.hd = cfg.d_model // cfg.n_heads
        self.groups = self.h // self.kv
        self.q_proj = nn.Linear(cfg.d_model, self.h * self.hd, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, self.kv * self.hd, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, self.kv * self.hd, bias=False)
        self.o_proj = nn.Linear(self.h * self.hd, cfg.d_model, bias=False)

    def _qkv(self, x):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        return q, k, v

    def forward(self, x, cos, sin):
        B, T, _ = x.shape
        q, k, v = self._qkv(x)
        q = M3_apply_rope(q, cos, sin)
        k = M3_apply_rope(k, cos, sin)
        k, v = M3_repeat_kv(k, self.groups), M3_repeat_kv(v, self.groups)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, -1))


class M3_SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.w_gate = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.w_up = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.w_down = nn.Linear(cfg.ffn_hidden, cfg.d_model, bias=False)

    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))


class M3_Block(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.norm1 = M3_RMSNorm(cfg.d_model)
        self.attn = M3_GQAttention(cfg)
        self.norm2 = M3_RMSNorm(cfg.d_model)
        self.ffn = M3_SwiGLU(cfg)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, cos, sin):
        x = x + self.drop(self.attn(self.norm1(x), cos, sin))
        x = x + self.drop(self.ffn(self.norm2(x)))
        return x


class M3_MTPHead(nn.Module):
    

    def __init__(self, dim):
        super().__init__()
        self.proj = nn.Linear(dim, dim, bias=False)
        self.norm = M3_RMSNorm(dim)

    def forward(self, x):
        return self.norm(x + self.proj(x))


class M3_DecoderLM(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)       # also the (tied) output projection of every head
        self.blocks = nn.ModuleList([M3_Block(cfg) for _ in range(cfg.n_layers)])
        self.heads = nn.ModuleList([M3_MTPHead(cfg.d_model) for _ in range(cfg.mtp_heads)])
        self.drop = nn.Dropout(cfg.dropout)
        cos, sin = M3_build_rope(cfg.d_model // cfg.n_heads, cfg.max_len)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        self.apply(self._init_weights)
        # damp the residual-branch output projections (GPT-2 style)
        scale = 0.02 / math.sqrt(2 * cfg.n_layers)
        for name, p in self.named_parameters():
            if name.endswith("o_proj.weight") or name.endswith("w_down.weight"):
                nn.init.normal_(p, mean=0.0, std=scale)

        for h in self.heads:
            nn.init.zeros_(h.proj.weight)
        self._ban = None

    @staticmethod
    def _init_weights(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def _rope(self, T, pos_ids=None):
        if pos_ids is None:
            return self.rope_cos[:T][None, None], self.rope_sin[:T][None, None]
        return self.rope_cos[pos_ids][:, None], self.rope_sin[pos_ids][:, None]   # (B, 1, T, hd/2)

    def hidden(self, idx):
        """idx (B, T) -> residual stream (B, T, D) that all MTP heads read"""
        cos, sin = self._rope(idx.size(1))
        x = self.drop(self.embed(idx))
        for blk in self.blocks:
            x = blk(x, cos, sin)
        return x

    def head_z(self, x, k):
        """normalised features of MTP head k (0-based; head 0 = next token)"""
        return self.heads[k](x)

    def forward(self, idx):
        
        return F.linear(self.head_z(self.hidden(idx), 0), self.embed.weight)


class M4_RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight).type_as(x)


def M4_build_rope(head_dim, max_len, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    freqs = torch.outer(torch.arange(max_len).float(), inv_freq)
    return freqs.cos(), freqs.sin()          # each (max_len, head_dim/2)


def M4_apply_rope(x, cos, sin):
    # x: (B, H, T, hd); cos/sin: (1 or B, 1, T, hd/2)
    x1, x2 = x.float().chunk(2, dim=-1)
    out = torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.type_as(x)


def M4_repeat_kv(x, groups):
    # (B, KV, T, hd) -> (B, KV*groups, T, hd); query head h reads kv head h // groups
    if groups == 1:
        return x
    B, KV, T, hd = x.shape
    return x[:, :, None].expand(B, KV, groups, T, hd).reshape(B, KV * groups, T, hd)


M4__BLOCK_MASKS = {}


def M4_block_local_mask(window, n_blocks, device):
    
    key = (window, n_blocks, str(device))
    m = M4__BLOCK_MASKS.get(key)
    if m is None:
        i = torch.arange(window, device=device)[:, None]
        j = torch.arange(2 * window, device=device)[None, :]
        base = (j > i) & (j <= i + window)                   # (W, 2W)
        m = base[None].repeat(n_blocks, 1, 1)                # (nb, W, 2W)
        m[0] &= (j >= window)                                # no previous block in front of block 0
        m = m[:, None]
        M4__BLOCK_MASKS[key] = m
    return m


def M4_local_attention(q, k, v, window, dropout_p=0.0):
    
    B, H, T, hd = q.shape
    nb = (T + window - 1) // window
    Tp = nb * window
    if Tp != T:                          # padded keys sit after every real query, so causality hides them
        q, k, v = (F.pad(t, (0, 0, 0, Tp - T)) for t in (q, k, v))
    qb = q.reshape(B, H, nb, window, hd).transpose(1, 2).reshape(B * nb, H, window, hd)

    def two_blocks(t):                   # (B, H, Tp, hd) -> (B*nb, H, 2W, hd) = [previous block | own block]
        tb = t.reshape(B, H, nb, window, hd)
        prev = torch.cat([tb.new_zeros(B, H, 1, window, hd), tb[:, :, :-1]], dim=2)
        return torch.cat([prev, tb], dim=3).transpose(1, 2).reshape(B * nb, H, 2 * window, hd)

    mask = M4_block_local_mask(window, nb, q.device)                                    # (nb, 1, W, 2W)
    mask = mask[None].expand(B, nb, 1, window, 2 * window).reshape(B * nb, 1, window, 2 * window)
    out = F.scaled_dot_product_attention(qb, two_blocks(k), two_blocks(v), attn_mask=mask, dropout_p=dropout_p)
    return out.reshape(B, nb, H, window, hd).transpose(1, 2).reshape(B, H, Tp, hd)[:, :, :T]


def M4_local_attention_ref(q, k, v, window, dropout_p=0.0):
    
    T = q.size(2)
    i = torch.arange(T, device=q.device)
    mask = (i[None, :] <= i[:, None]) & (i[None, :] > i[:, None] - window)
    return F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=dropout_p)


M4_LOCAL_IMPL = {"mode": "block"}


class M4_GQAttention(nn.Module):

    

    def __init__(self, cfg, window):
        super().__init__()
        self.h = cfg.n_heads
        self.kv = cfg.n_kv_heads
        self.hd = cfg.d_model // cfg.n_heads
        self.groups = self.h // self.kv
        self.window = window
        self.p = cfg.attn_dropout
        self.q_proj = nn.Linear(cfg.d_model, self.h * self.hd, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, self.kv * self.hd, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, self.kv * self.hd, bias=False)
        self.o_proj = nn.Linear(self.h * self.hd, cfg.d_model, bias=False)

    def _qkv(self, x):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        return q, k, v

    def forward(self, x, cos, sin):
        B, T, _ = x.shape
        q, k, v = self._qkv(x)
        q = M4_apply_rope(q, cos, sin)
        k = M4_apply_rope(k, cos, sin)
        k, v = M4_repeat_kv(k, self.groups), M4_repeat_kv(v, self.groups)
        p = self.p if self.training else 0.0
        attend = M4_local_attention if M4_LOCAL_IMPL["mode"] == "block" else M4_local_attention_ref
        out = attend(q, k, v, self.window, p)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, -1))


class M4_SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.w_gate = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.w_up = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.w_down = nn.Linear(cfg.ffn_hidden, cfg.d_model, bias=False)

    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))


class M4_LocalBlock(nn.Module):

    """pre-norm block: sliding-window GQA attention (RoPE) + M4_SwiGLU"""

    kind = "local"

    def __init__(self, cfg, window):
        super().__init__()
        self.norm1 = M4_RMSNorm(cfg.d_model)
        self.attn = M4_GQAttention(cfg, window)
        self.norm2 = M4_RMSNorm(cfg.d_model)
        self.ffn = M4_SwiGLU(cfg)
        self.drop = nn.Dropout(cfg.ffn_dropout)

    def forward(self, x, cos, sin):
        x = x + self.attn(self.norm1(x), cos, sin)
        x = x + self.drop(self.ffn(self.norm2(x)))
        return x


def M4_memory_masks(q_chunk, k_chunk, k_valid, mem_per_chunk, n_slots):
    
    slot_chunk = torch.arange(n_slots, device=q_chunk.device) // mem_per_chunk                 # (slots,)
    w = (k_chunk[:, None, :] == slot_chunk[None, :, None]) & k_valid[:, None, :]               # (Bk, slots, S)
    w = torch.where(w.any(-1, keepdim=True), w, k_valid[:, None, :])
    r = slot_chunk[None, None, :] < q_chunk[:, :, None]                                        # (Bq, Tq, slots)
    has = r.any(-1)
    r = torch.where(has[..., None], r, torch.ones_like(r))
    return w[:, None], r[:, None], has


class M4_MemoryBlock(nn.Module):

    

    kind = "memory"

    def __init__(self, cfg):
        super().__init__()
        d = cfg.d_model
        self.h = cfg.n_heads
        self.kv = cfg.n_kv_heads
        self.hd = d // self.h
        self.groups = self.h // self.kv
        self.p = cfg.attn_dropout
        self.norm1 = M4_RMSNorm(d)
        self.norm2 = M4_RMSNorm(d)
        self.mem_norm_w = M4_RMSNorm(d)                 # norm of the memory as query of the write attention
        self.mem_norm_r = M4_RMSNorm(d)                 # norm of the memory as key/value of the read attention
        # write: memory tokens (queries) <- chunk tokens (keys / values), no RoPE
        self.wq_proj = nn.Linear(d, self.h * self.hd, bias=False)
        self.wk_proj = nn.Linear(d, self.kv * self.hd, bias=False)
        self.wv_proj = nn.Linear(d, self.kv * self.hd, bias=False)
        self.write_out = nn.Linear(self.h * self.hd, d, bias=False)
        self.gate_proj = nn.Linear(2 * d, d, bias=False)            # g = sigmoid(W [M_prev ; candidate])
        # read: tokens (queries) <- memory tokens (keys / values), no RoPE
        self.q_proj = nn.Linear(d, self.h * self.hd, bias=False)
        self.k_proj = nn.Linear(d, self.kv * self.hd, bias=False)
        self.v_proj = nn.Linear(d, self.kv * self.hd, bias=False)
        self.o_proj = nn.Linear(self.h * self.hd, d, bias=False)
        self.ffn = M4_SwiGLU(cfg)
        self.drop = nn.Dropout(cfg.ffn_dropout)

    def write_kv(self, xn):
        B, T, _ = xn.shape
        k = self.wk_proj(xn).view(B, T, self.kv, self.hd).transpose(1, 2)
        v = self.wv_proj(xn).view(B, T, self.kv, self.hd).transpose(1, 2)
        return k, v

    def update(self, mem_prev, k_all, v_all, w_mask):
        
        B, n, _ = mem_prev.shape
        q = self.wq_proj(self.mem_norm_w(mem_prev)).view(B, n, self.h, self.hd).transpose(1, 2)
        keys = M4_repeat_kv(k_all, self.groups).to(q.dtype)
        vals = M4_repeat_kv(v_all, self.groups).to(q.dtype)
        out = F.scaled_dot_product_attention(q, keys, vals, attn_mask=w_mask, dropout_p=self.p if self.training else 0.0)
        cand = self.write_out(out.transpose(1, 2).reshape(B, n, -1)).type_as(mem_prev)
        gate = torch.sigmoid(self.gate_proj(torch.cat([mem_prev, cand], dim=-1)))
        return mem_prev + gate * (cand - mem_prev)                    # = (1 - g) * M_prev + g * candidate

    def read(self, xn, mem, r_mask, has):
        B, T, _ = xn.shape
        n = mem.size(1)
        mn = self.mem_norm_r(mem)
        q = self.q_proj(xn).view(B, T, self.h, self.hd).transpose(1, 2)
        k = self.k_proj(mn).view(B, n, self.kv, self.hd).transpose(1, 2)
        v = self.v_proj(mn).view(B, n, self.kv, self.hd).transpose(1, 2)
        out = F.scaled_dot_product_attention(q, M4_repeat_kv(k, self.groups), M4_repeat_kv(v, self.groups), attn_mask=r_mask,
                                             dropout_p=self.p if self.training else 0.0)
        out = self.o_proj(out.transpose(1, 2).reshape(B, T, -1))
        return out * has[..., None].to(out.dtype)

    def forward(self, x, mem_prev, mctx):
        xn = self.norm1(x)
        k, v = self.write_kv(xn)
        mem = self.update(mem_prev, k, v, mctx["w"])
        x = x + self.read(xn, mem, mctx["r"], mctx["has"])
        x = x + self.drop(self.ffn(self.norm2(x)))
        return x, mem


class M4_DecoderLM(nn.Module):

    

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.n_slots = cfg.n_mem_chunks * cfg.mem_per_chunk
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)       # also the (tied) output projection
        windows = {"local_small": cfg.window_small, "local_large": cfg.window_large}
        self.blocks = nn.ModuleList([M4_MemoryBlock(cfg) if kind == "memory" else M4_LocalBlock(cfg, windows[kind])
                                     for kind in cfg.layer_types])
        self.final_norm = M4_RMSNorm(cfg.d_model)
        self.mem_init = nn.Parameter(torch.empty(self.n_slots, cfg.d_model))      # learned latent memory
        self.mtp_proj = nn.ModuleList([nn.Linear(cfg.d_model, cfg.d_model, bias=False) for _ in range(cfg.n_mtp - 1)])
        self.mtp_norm = nn.ModuleList([M4_RMSNorm(cfg.d_model) for _ in range(cfg.n_mtp - 1)])
        cos, sin = M4_build_rope(cfg.d_model // cfg.n_heads, cfg.max_len)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        self.apply(self._init_weights)
        nn.init.normal_(self.mem_init, mean=0.0, std=0.02)
        # damp the residual-branch output projections (GPT-2 style)
        scale = 0.02 / math.sqrt(2 * cfg.n_layers)
        for name, p in self.named_parameters():
            if name.endswith("o_proj.weight") or name.endswith("w_down.weight"):
                nn.init.normal_(p, mean=0.0, std=scale)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def _rope(self, T, pos_ids=None):
        if pos_ids is None:
            return self.rope_cos[:T][None, None], self.rope_sin[:T][None, None]
        return self.rope_cos[pos_ids][:, None], self.rope_sin[pos_ids][:, None]   # (B, 1, T, hd/2)

    def _chunk_of(self, pos):
        c = self.cfg
        return (pos // c.chunk_size).clamp(max=c.n_mem_chunks - 1)

    def _mem_ctx_train(self, T, device):
        pos = torch.arange(T, device=device)[None]                  # (1, T): no padding in training
        chunk = self._chunk_of(pos)
        w, r, has = M4_memory_masks(chunk, chunk, torch.ones_like(pos, dtype=torch.bool), self.cfg.mem_per_chunk, self.n_slots)
        return {"w": w, "r": r, "has": has}

    def trunk(self, idx):
        """(B, T) ids -> (B, T, D) final hidden states"""
        B, T = idx.shape
        cos, sin = self._rope(T)
        x = self.embed(idx)
        mem, mctx = None, None
        for blk in self.blocks:
            if blk.kind == "memory":
                if mem is None:
                    mem = self.mem_init[None].expand(B, -1, -1)       # the learned latent memory enters the first memory layer
                    mctx = self._mem_ctx_train(T, idx.device)
                x, mem = blk(x, mem, mctx)
            else:
                x = blk(x, cos, sin)
        return self.final_norm(x)

    def head_input(self, h, k):
        """hidden state as seen by head k (0-based: head 0 predicts the next token)"""
        if k == 0:
            return h
        return self.mtp_norm[k - 1](h + self.mtp_proj[k - 1](h))

    def forward(self, idx):
        """training / validation path of the next-token head: idx (B, T) -> logits (B, T, V)"""
        return F.linear(self.head_input(self.trunk(idx), 0), self.embed.weight)      # weight tying: logits = h @ E^T


@torch.no_grad()
def check_local_attention(cfg):
    
    g = torch.Generator().manual_seed(0)
    worst = 0.0
    for window in (cfg.window_small, cfg.window_large):
        for T in (window - 5, 300, 640):
            q, k, v = (torch.randn(2, 4, T, 16, generator=g).to(DEVICE) for _ in range(3))
            diff = (M4_local_attention(q, k, v, window) - M4_local_attention_ref(q, k, v, window)).abs().max().item()
            worst = max(worst, diff)
    if worst > 1e-4:
        M4_LOCAL_IMPL["mode"] = "mask"
        log(f"WARNING: block-local attention differs from the masked reference by {worst:.2e} -> using masked SDPA instead")
    else:
        log(f"self-test passed: block-local attention == masked reference (max diff {worst:.2e})")


P1 = {}
BACKBONES = {}


def get_backbone(which):
    
    if which in BACKBONES:
        return BACKBONES[which]
    info = P1[which]
    ck = info["ck"]
    assert "model" in ck, f"{which}: Phase-1 weights were already consumed"
    sd = {re.sub(r"^(module\.|_orig_mod\.)+", "", k): v for k, v in ck["model"].items()}
    Cfg, LM = (M3_ModelConfig, M3_DecoderLM) if which == "teacher" else (M4_ModelConfig, M4_DecoderLM)
    cfg = Cfg(**{k: v for k, v in info["cfg"].items() if k in Cfg.__dataclass_fields__})
    lm = LM(cfg)
    lm.load_state_dict(sd, strict=True)          # fails loudly if architecture / keys differ
    assert CHUNK_LEN <= cfg.max_len, f"CHUNK_LEN {CHUNK_LEN} exceeds the RoPE table of the {which} ({cfg.max_len})"
    ck.pop("model", None)
    ck.pop("optimizer", None)
    del sd
    gc.collect()
    lm = lm.to(DEVICE)
    if which == "student":
        check_local_attention(cfg)
        log(f"student backbone (Model 4): {cfg.n_layers} layers {list(cfg.layer_types)}, d_model {cfg.d_model}, ffn {cfg.ffn_hidden}, "
            f"{cfg.n_heads}q/{cfg.n_kv_heads}kv heads, {cfg.n_mem_chunks}x{cfg.mem_per_chunk} memory tokens, vocab rows {cfg.vocab_size}")
    else:
        log(f"teacher backbone (Model 3): {cfg.n_layers} layers, d_model {cfg.d_model}, ffn {cfg.ffn_hidden}, "
            f"{cfg.n_heads}q/{cfg.n_kv_heads}kv heads, {cfg.mtp_heads} MTP heads, vocab rows {cfg.vocab_size}")
    BACKBONES[which] = lm
    return lm


def teacher_feats(lm, ids):
    """Model 3: MTP head 1 = RMSNorm(x + W x) on the residual stream"""
    return lm.head_z(lm.hidden(ids), 0)


def student_feats(lm, ids):
    """Model 4: final hidden states after final_norm (local + memory layers)"""
    return lm.trunk(ids)


# ---------------------------- LoRA ----------------------------
class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha, dropout, rs=False):
        super().__init__()
        self.base = base
        self.scale = alpha / (math.sqrt(r) if rs else r)
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))      # B = 0  ->  starts exactly at the pretrained model
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(self.drop(x), self.A), self.B) * self.scale


def inject_lora(lm, targets, r, alpha, dropout, rs, ft_dropout):
    
    for m in lm.modules():
        if isinstance(m, nn.Dropout):
            m.p = ft_dropout
    for p in lm.parameters():
        p.requires_grad = False
    n = 0

    def wrap(module):
        nonlocal n
        for name, child in list(module.named_children()):
            if isinstance(child, nn.Linear) and name in targets:
                setattr(module, name, LoRALinear(child, r, alpha, dropout, rs))
                n += 1
            elif not isinstance(child, LoRALinear):
                wrap(child)

    wrap(lm)
    return n


class CkptBlock(nn.Module):
    

    def __init__(self, blk, enabled):
        super().__init__()
        self.blk = blk
        self.kind = blk.kind
        self.enabled = enabled

    def forward(self, *args):
        if self.enabled and self.training and torch.is_grad_enabled():
            return checkpoint(self.blk, *args, use_reentrant=False)
        return self.blk(*args)


PATH_NAMES = ("attention", "chunk-LSE", "mean")


class LSIModel(nn.Module):
    

    def __init__(self, lm, feat_fn, n_labels, prior_logits, hp):
        super().__init__()
        self.lm = lm
        self.feat_fn = feat_fn
        self.hp = dict(hp)
        d, H = lm.cfg.d_model, hp["hidden"]
        self.chunk_proj = nn.Sequential(nn.LayerNorm(2 * d), nn.Dropout(hp["dropout"]),
                                        nn.Linear(2 * d, H), nn.GELU())
        self.pos_emb = nn.Embedding(hp["max_pos"], H)             # absolute chunk index
        self.rel_emb = nn.Embedding(hp["rel_buckets"] + 1, H)     # relative position bucket (+1: single-chunk document)
        nn.init.normal_(self.pos_emb.weight, std=0.02)
        nn.init.normal_(self.rel_emb.weight, std=0.02)
        layer = nn.TransformerEncoderLayer(d_model=H, nhead=hp["heads"], dim_feedforward=hp["ff"], dropout=hp["dropout"],
                                           activation="gelu", batch_first=True, norm_first=True)
        self.doc_enc = nn.TransformerEncoder(layer, num_layers=hp["layers"], enable_nested_tensor=False)
        self.doc_norm = nn.LayerNorm(H)
        self.Q = nn.Parameter(torch.randn(n_labels, H) * 0.1)              # (a) label queries over chunks
        self.W_att = nn.Parameter(torch.randn(n_labels, H) * 0.02)
        self.W_chunk = nn.Parameter(torch.randn(n_labels, H) * 0.02)       # (b) chunk-level statute classifier
        self.W_mean = nn.Parameter(torch.randn(n_labels, H) * 0.02)        # (c)
        self.gate = nn.Parameter(torch.zeros(n_labels, 3))                 # softmax -> equal mix at init
        self.b = nn.Parameter(torch.as_tensor(prior_logits, dtype=torch.float32))   # init = log prior odds


    def _encode_mb(self, ids, lens):
        N, T = ids.shape
        x = self.feat_fn(self.lm, ids).float()                # (N, T, d)

        mask = (torch.arange(T, device=ids.device)[None, :] < lens[:, None]).unsqueeze(-1)
        mean = (x * mask).sum(1) / lens[:, None].float()
        last = x[torch.arange(N, device=ids.device), lens - 1]
        return torch.cat([mean, last], dim=-1)

    def encode_chunks(self, ids, lens):
        """ALL chunks of the batch. Processed in length-sorted micro-batches purely to bound memory."""
        N = ids.size(0)
        mb = ENC_MB_TRAIN if self.training else ENC_MB_EVAL
        if N <= mb:
            return self._encode_mb(ids, lens)
        order = torch.argsort(lens, descending=True)
        outs = []
        for i in range(0, N, mb):
            sel = order[i:i + mb]
            T = int(lens[sel].max().item())
            outs.append(self._encode_mb(ids[sel, :T], lens[sel]))
        out = torch.cat(outs, 0)
        inv = torch.empty_like(order)
        inv[order] = torch.arange(N, device=order.device)
        return out[inv]


    def forward(self, ids, lens, owner, pos, n_docs):
        
        emb = self.encode_chunks(ids, lens)                                   # (N, 2d) fp32
        with no_amp():
            u = self.chunk_proj(emb.float())                                  # (N, H)
            H = u.size(-1)
            counts = torch.bincount(owner, minlength=n_docs)                  # chunks per doc
            Cmax = int(counts.max().item())
            ncd = counts[owner]                                               # chunks of the doc each chunk belongs to
            rbk = self.hp["rel_buckets"]
            rb = (((pos.float() + 0.5) / ncd.float()) * rbk).long().clamp(0, rbk - 1)
            rb = torch.where(ncd == 1, torch.full_like(rb, rbk), rb)
            u = u + self.pos_emb(pos.clamp(max=self.hp["max_pos"] - 1)) + self.rel_emb(rb)

            U = u.new_zeros(n_docs, Cmax, H).index_put((owner, pos), u)       # (D, C, H) padded
            valid = torch.zeros(n_docs, Cmax, dtype=torch.bool, device=u.device)
            valid[owner, pos] = True
            vm = valid[:, :, None]
            V = self.doc_norm(self.doc_enc(U, src_key_padding_mask=~valid))   # chunks attend to each other
            V = V.masked_fill(~vm, 0.0)                                       # padded slots can never leak (0 * NaN = NaN)

            nv = valid.sum(1).float()                                         # (D,)  real chunks per doc
            # (a) label-wise attention over chunks
            sc = torch.einsum("dch,lh->dcl", V, self.Q) / math.sqrt(H)
            alpha = torch.softmax(sc.masked_fill(~vm, float("-inf")), dim=1)  # (D, C, L)
            z_att = (torch.einsum("dcl,dch->dlh", alpha, V) * self.W_att).sum(-1)
            # (b) chunk-level statute logits, log-mean-exp over the real chunks
            s_raw = torch.einsum("dch,lh->dcl", V, self.W_chunk)              # (D, C, L), exactly 0 on padded chunks
            z_pool = torch.logsumexp(s_raw.masked_fill(~vm, float("-inf")), dim=1) - torch.log(nv)[:, None]
            # (c) mean of chunk representations
            z_mean = ((V * vm).sum(1) / nv[:, None]) @ self.W_mean.t()

            zs = torch.stack([z_att, z_pool, z_mean], dim=-1)                 # (D, L, 3)
            g = torch.softmax(self.gate, dim=-1)                              # (L, 3)
            z = (zs * g[None]).sum(-1) + self.b
            return z, zs + self.b[None, :, None], {"alpha": alpha, "s": s_raw, "valid": valid}


def infer_head_hp(sd):
    
    layers = 1 + max(int(m.group(1)) for k in sd for m in [re.match(r"doc_enc\.layers\.(\d+)\.", k)] if m)
    return dict(hidden=int(sd["chunk_proj.2.weight"].shape[0]), dropout=TEACHER_HEAD_DROPOUT, layers=layers,
                heads=TEACHER_DOC_HEADS, ff=int(sd["doc_enc.layers.0.linear1.weight"].shape[0]),
                max_pos=int(sd["pos_emb.weight"].shape[0]), rel_buckets=int(sd["rel_emb.weight"].shape[0]) - 1)


def trainable_state(model):
    return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}


def load_trainable(model, sd):
    missing, unexpected = model.load_state_dict(sd, strict=False)
    assert not unexpected, f"unexpected keys in checkpoint: {unexpected[:5]}"
    train_names = {n for n, p in model.named_parameters() if p.requires_grad}
    assert not (train_names & set(missing)), f"some trainable tensors were not restored: {sorted(train_names & set(missing))[:5]}"


def build_student():
    
    lm = get_backbone("student")
    n_lora = inject_lora(lm, STUDENT_LORA_TARGETS, LORA_R, LORA_ALPHA, LORA_DROPOUT, USE_RSLORA, FT_DROPOUT)
    lm.final_norm.weight.requires_grad = True
    lm.mem_init.requires_grad = STUDENT_TRAIN_MEM_INIT
    lm.blocks = nn.ModuleList([CkptBlock(b, GRAD_CKPT) for b in lm.blocks])
    prior = np.clip(Y_train.mean(0), 1e-4, 1 - 1e-4)
    hp = dict(hidden=HEAD_HIDDEN, dropout=HEAD_DROPOUT, layers=DOC_LAYERS, heads=DOC_HEADS, ff=DOC_FF,
              max_pos=MAX_POS_EMB, rel_buckets=REL_BUCKETS)
    model = LSIModel(lm, student_feats, N_LABELS, np.log(prior / (1 - prior)), hp).to(DEVICE)
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_all = sum(p.numel() for p in model.parameters())
    log(f"student LoRA modules: {n_lora} | trainable {n_tr / 1e6:.2f}M / total {n_all / 1e6:.1f}M params ({100 * n_tr / n_all:.1f}%)")
    return model


In [3]:
# CELL 3: CHECKPOINTS AND DATA

from huggingface_hub import hf_hub_download, login


def load_phase1(which, tag, override):
    
    path = find_phase1_ckpt(tag, override)
    ck = torch.load(path, map_location="cpu", weights_only=False)
    keys = [re.sub(r"^(module\.|_orig_mod\.)+", "", k) for k in ck["model"].keys()]
    is_m4 = "mem_init" in keys and "final_norm.weight" in keys
    is_m3 = any(k.startswith("heads.0.") for k in keys) and "final_norm.weight" not in keys
    if which == "teacher" and not is_m3:
        raise ValueError(f"{path}: the TEACHER must be Model 3 (RoPE+GQA+SwiGLU+MTP, MTP heads, no final_norm / mem_init) - "
                         f"this checkpoint looks like {'Model 4 (memory)' if is_m4 else 'another model'}. Check TEACHER_P1_TAG / TEACHER_P1_CKPT.")
    if which == "student" and not is_m4:
        raise ValueError(f"{path}: the STUDENT must be Model 4 (MSM transformer with mem_init + final_norm) - "
                         f"this checkpoint looks like {'Model 3 (MTP)' if is_m3 else 'another model'}. Check STUDENT_P1_TAG / STUDENT_P1_CKPT.")
    cfg = ck.get("config") or {}
    P1[which] = {"path": path, "ck": ck, "cfg": cfg, "val_loss": ck.get("val_loss"), "step": ck.get("global_step")}
    log(f"Phase-1 {which:7s}: {path}")
    log(f"   config: d_model {cfg.get('d_model')} | vocab rows {cfg.get('vocab_size')} | max_len {cfg.get('max_len')} | "
        f"pretraining val_loss {ck.get('val_loss')} | global_step {ck.get('global_step')}")
    if not cfg:
        raise ValueError(f"{path} has no saved 'config' - cannot rebuild its architecture")


if USE_TEACHER:
    load_phase1("teacher", TEACHER_P1_TAG, TEACHER_P1_CKPT)
load_phase1("student", STUDENT_P1_TAG, STUDENT_P1_CKPT)

# ---- Phase-5 data from Hugging Face ----
HF_TOKEN = get_hf_token()
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN)


def load_records(split):
    path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=PHASE5_FILES[split],
                           repo_type="dataset", token=HF_TOKEN)
    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    if isinstance(obj, dict):
        for key in ("data", "records", "examples"):
            if isinstance(obj.get(key), list):
                obj = obj[key]
                break
    assert isinstance(obj, list), f"unexpected JSON structure in {split}"
    log(f"{split:5s}: {len(obj):,} documents")
    return obj


raw = {s: load_records(s) for s in ("train", "dev", "test")}


def parse_labels(lst):
    out = set()
    for x in lst:
        if isinstance(x, bool):
            continue
        if isinstance(x, (int, np.integer)):
            out.add(int(x))
        elif isinstance(x, str) and x.strip().isdigit():
            out.add(int(x.strip()))
    return sorted(out)          # sorted + unique  (3 train docs / 1 dev doc contain duplicate labels)


# label space = every statute id seen in train or dev (100 ids in your EDA)
LABEL_IDS = sorted({l for s in ("train", "dev") for r in raw[s] for l in parse_labels(r["labels"])})
LAB2IDX = {l: i for i, l in enumerate(LABEL_IDS)}
N_LABELS = len(LABEL_IDS)
log(f"{N_LABELS} statutes | id range {LABEL_IDS[0]}..{LABEL_IDS[-1]}")


TEACHER_DIR, TEACHER_META, TEACHER_LSI_PATH = (None, None, None)
if USE_TEACHER:
    TEACHER_DIR, TEACHER_META = find_teacher_lsi()
    TEACHER_LSI_PATH = TEACHER_LSI_CKPT or os.path.join(TEACHER_DIR, "lsi_best.pt")
    if TEACHER_META.get("label_ids") != LABEL_IDS:
        raise ValueError("the fine-tuned teacher was trained on a different label space than this data "
                         f"({TEACHER_META.get('n_labels')} vs {N_LABELS} statutes) - it cannot be distilled into this student")
    log(f"fine-tuned TEACHER: {TEACHER_LSI_PATH} | dev macro-F1 {TEACHER_META['dev_macro_f1']:.4f} "
        f"({TEACHER_META.get('where')}, step {TEACHER_META.get('global_step')})")

if SMOKE_TEST:                  # tiny deterministic subsets (labels were computed on the full data above)
    rs = random.Random(SEED)
    for s in raw:
        keep = sorted(rs.sample(range(len(raw[s])), min(N_SMOKE[s], len(raw[s]))))
        raw[s] = [raw[s][i] for i in keep]
        log(f"SMOKE: {s} -> {len(raw[s])} docs")


def build_Y(recs):
    Y = np.zeros((len(recs), N_LABELS), dtype=np.uint8)
    for i, r in enumerate(recs):
        for l in parse_labels(r["labels"]):
            Y[i, LAB2IDX[l]] = 1
    return Y


Y_train, Y_dev = build_Y(raw["train"]), build_Y(raw["dev"])

# ---- label audit: class imbalance / rare statutes / co-occurrence ----
_pos = Y_train.sum(0)
_q = np.percentile(_pos, [0, 25, 50, 75, 100])
log(f"labels/doc: train {Y_train.sum(1).mean():.2f} | dev {Y_dev.sum(1).mean():.2f} | "
    f"single-label docs (train): {(Y_train.sum(1) == 1).mean():.1%}")
log(f"train positives per statute: min {_q[0]:.0f} | p25 {_q[1]:.0f} | median {_q[2]:.0f} | p75 {_q[3]:.0f} | max {_q[4]:.0f}")
log(f"rare statutes (train positives): <10: {int((_pos < 10).sum())} | <25: {int((_pos < 25).sum())} | "
    f"<50: {int((_pos < 50).sum())}  (macro-F1 weights each of them like the frequent ones)")
_o = np.argsort(-_pos)
log("most frequent:", ", ".join(f"{LABEL_IDS[i]}({int(_pos[i])})" for i in _o[:5]),
    "| rarest:", ", ".join(f"{LABEL_IDS[i]}({int(_pos[i])})" for i in _o[-5:]))
_C = Y_train.T.astype(np.int64) @ Y_train.astype(np.int64)
_iu = np.triu_indices(N_LABELS, 1)
_top = np.argsort(-_C[_iu])[:5]
log("top co-occurring statute pairs:", ", ".join(f"({LABEL_IDS[_iu[0][t]]},{LABEL_IDS[_iu[1][t]]}):{int(_C[_iu][t])}" for t in _top))
log(f"statutes with 0 dev positives: {int((Y_dev.sum(0) == 0).sum())}")


Phase-1 teacher: /kaggle/input/datasets/krishkalg/model3-mtp/cs781_phase1_model3/checkpoints/best.pt
   config: d_model 640 | vocab rows 50048 | max_len 1088 | pretraining val_loss 2.352539456367493 | global_step 50000
Phase-1 student: /kaggle/input/datasets/krishkalgude/version1/cs781_phase1_model4/checkpoints/best.pt
   config: d_model 512 | vocab rows 50000 | max_len 1088 | pretraining val_loss 2.3764375047683717 | global_step 50000


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

train: 42,750 documents


Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

dev  : 10,181 documents


Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

test : 13,019 documents
100 statutes | id range 0..99
fine-tuned TEACHER: /kaggle/input/datasets/krishkalgude215/p5-m3-lorma/p5_lsi_m3_lora_v1_allchunks/checkpoints/lsi_best.pt | dev macro-F1 0.4063 (epoch4, step 12946)
labels/doc: train 3.70 | dev 3.92 | single-label docs (train): 21.4%
train positives per statute: min 272 | p25 472 | median 679 | p75 1751 | max 11730
rare statutes (train positives): <10: 0 | <25: 0 | <50: 0  (macro-F1 weights each of them like the frequent ones)
most frequent: 5(11730), 34(10380), 97(7880), 43(7728), 39(7120) | rarest: 66(309), 92(309), 23(304), 25(287), 84(272)
top co-occurring statute pairs: (5,34):3644, (12,13):3511, (43,97):3385, (5,43):3271, (13,14):2831
statutes with 0 dev positives: 0


In [4]:
# CELL 4: TOKENIZER AND TOKEN CACHE

WORD_PATTERN = re.compile(
    r"""'s|'t|'re|'ve|'m|'ll|'d|[a-zA-Z]+|[0-9]+|[^\s\a-zA-Z0-9]+"""
)


class LegalBPE:
    def __init__(self, vocab_path, merges_path, meta_path=None):
        # Load vocabulary
        with open(vocab_path, "r", encoding="utf-8") as f:
            self.vocab = json.load(f)

        # Load merge rules
        with open(merges_path, "r", encoding="utf-8") as f:
            merges = json.load(f)

         # Tokenizer fingerprint for window-cache validation
        h = hashlib.md5()

        for path in (vocab_path, merges_path, meta_path):
            if path and os.path.exists(path):
                with open(path, "rb") as f:
                    h.update(f.read())

        self.fingerprint = h.hexdigest()[:12]

        
        # Merge rank: lower rank = earlier BPE merge
        self.ranks = {
            (a, b): i
            for i, (a, b) in enumerate(merges)
        }

        # ID -> token
        self.inv = {
            int(i): t
            for t, i in self.vocab.items()
        }

        self.vocab_size = max(self.inv) + 1

        assert len(self.inv) == self.vocab_size, \
            "vocab ids are not contiguous"

        self.pad_id = self.vocab["<|pad|>"]
        self.unk_id = self.vocab["<|unk|>"]
        self.eos_id = self.vocab["<|endoftext|>"]
        self.bos_id = self.eos_id
        self.mask_id = self.eos_id

        self.special_ids = {
            self.pad_id,
            self.unk_id,
            self.eos_id,
        }
        # Your tokenizer metadata
        meta = {}
        if meta_path and os.path.exists(meta_path):
            with open(meta_path, "r", encoding="utf-8") as f:
                meta = json.load(f)

        for key, val in (
            ("pad_id", self.pad_id),
            ("unk_id", self.unk_id),
            ("eos_id", self.eos_id),
            ("bos_id", self.bos_id),
            ("mask_id", self.mask_id),
        ):
            if key in meta and meta[key] != val:
                raise ValueError(
                    f"meta.json {key}={meta[key]} disagrees with vocab ({val})"
            )

        if meta.get("vocab_size") not in (None, self.vocab_size):
            log(
                f"warning: meta.json vocab_size="
                f"{meta['vocab_size']} but vocab has "
                f"{self.vocab_size}"
            )

        # Decoder-only model
        self.bos_id = self.eos_id
        self.special_ids = {
            self.pad_id,
            self.unk_id,
            self.eos_id,
        }

        self.is_end = [
            self.inv[i].endswith("</w>")
            for i in range(self.vocab_size)
        ]

        # Cache BPE results for repeated words
        self.cache = {}

    def _word_ids(self, word):
        hit = self.cache.get(word)
        if hit is not None:
            return hit

        # Character sequence + word-end marker
        syms = list(word) + ["</w>"]

        # Apply BPE merges
        while len(syms) > 1:
            best = None
            best_rank = None

            for i in range(len(syms) - 1):
                r = self.ranks.get(
                    (syms[i], syms[i + 1])
                )

                if r is not None and (
                    best_rank is None or r < best_rank
                ):
                    best = (
                        syms[i],
                        syms[i + 1]
                    )
                    best_rank = r

            if best is None:
                break

            merged = best[0] + best[1]

            out = []
            i = 0

            while i < len(syms):
                if (
                    i < len(syms) - 1
                    and syms[i] == best[0]
                    and syms[i + 1] == best[1]
                ):
                    out.append(merged)
                    i += 2
                else:
                    out.append(syms[i])
                    i += 1

            syms = out

        ids = [
            self.vocab.get(s, self.unk_id)
            for s in syms
        ]

        # Prevent unlimited cache growth
        if len(self.cache) > 3_000_000:
            self.cache.clear()

        self.cache[word] = ids
        return ids

    def encode(self, text):
        ids = []

        for w in WORD_PATTERN.findall(text.lower()):
            ids.extend(self._word_ids(w))

        return ids

    def encode_batch(self, texts):
        return [self.encode(t) for t in texts]

    def decode(self, ids):
        toks = [
            self.inv.get(int(i), "")
            for i in ids
            if int(i) not in self.special_ids
        ]

        return "".join(toks).replace("</w>", " ").strip()


class CustomWordBPE(LegalBPE):
    kind = "custom-word-BPE"

    def encode_batch(self, texts):
        return [self.encode(t) for t in texts]


class HFByteBPE:
    kind = "hf-byte-BPE"

    def __init__(self, path):
        try:
            from tokenizers import Tokenizer
        except ImportError:
            os.system("pip -q install tokenizers")
            from tokenizers import Tokenizer
        self.tokenizer = Tokenizer.from_file(path)
        v = self.tokenizer.get_vocab()
        self.vocab_size = self.tokenizer.get_vocab_size()
        self.pad_id, self.unk_id, self.eos_id = v["<|pad|>"], v["<|unk|>"], v["<|endoftext|>"]
        with open(path, "rb") as f:
            self.fingerprint = hashlib.md5(f.read()).hexdigest()[:12]

    def encode_batch(self, texts):
        return [e.ids for e in self.tokenizer.encode_batch(texts)]


def _file_has_wordend_marker(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            head = f.read(2_000_000)
        return "</w>" in head
    except Exception:
        return False


def discover_tokenizers(max_rows):
    roots = [TOKENIZER_DIR] if TOKENIZER_DIR else [INPUT_ROOT] + ([] if IS_KAGGLE else ["."])
    found, seen = [], set()
    for root in roots:
        if not os.path.isdir(root):
            continue
        for d, _, files in os.walk(root):
            fs = set(files)
            real = os.path.realpath(d)
            if real in seen:
                continue
            seen.add(real)
            if "vocab_50k.json" in fs and "merges_50k.json" in fs:
                found.append(("custom", d, "vocab_50k.json", "merges_50k.json", "meta_50k.json"))
            elif "vocab.json" in fs and "merges.json" in fs and _file_has_wordend_marker(os.path.join(d, "vocab.json")):
                found.append(("custom", d, "vocab.json", "merges.json", "meta.json"))
            if "tokenizer.json" in fs:
                found.append(("hf", d, "tokenizer.json", None, None))
    cands = []
    for kind, d, a, b, c in found:
        if TOKENIZER_KIND != "auto" and TOKENIZER_KIND != kind:
            continue
        try:
            if kind == "custom":
                t = CustomWordBPE(os.path.join(d, a), os.path.join(d, b), os.path.join(d, c))
            else:
                t = HFByteBPE(os.path.join(d, a))
        except Exception as e:
            log(f"  skip {kind} tokenizer in {d}: {type(e).__name__}: {str(e)[:120]}")
            continue
        if t.vocab_size > max_rows or t.vocab_size < 1000:
            log(f"  skip {t.kind} in {d}: vocab {t.vocab_size} does not fit the model's {max_rows} embedding rows")
            continue
        t.where = d
        cands.append(t)
    return cands


@torch.no_grad()
def lm_next_token_loss(lm, id_lists):
    
    lm.eval()
    tot, cnt = 0.0, 0
    rows = lm.cfg.vocab_size
    for ids in id_lists:
        if len(ids) < CHUNK_LEN + 1:
            continue
        if max(ids[:CHUNK_LEN + 1]) >= rows:
            return float("inf"), 0
        x = torch.tensor(ids[:CHUNK_LEN], dtype=torch.long, device=DEVICE)[None]
        y = torch.tensor(ids[1:CHUNK_LEN + 1], dtype=torch.long, device=DEVICE)
        with amp_ctx():
            logits = lm(x)
        tot += F.cross_entropy(logits[0].float(), y, reduction="sum").item()
        cnt += y.numel()
    return (tot / cnt if cnt else float("nan")), cnt


_PH = re.compile(r"<([A-Z][A-Z_]*)>")
_WS = re.compile(r"\s+")


def prep_text(t):
    if not isinstance(t, str):
        t = ""
    if SIMPLIFY_PLACEHOLDERS:
        t = _PH.sub(r"\1", t)
    return _WS.sub(" ", t).strip()


def models_to_check():
    return [w for w in ("teacher", "student") if w in P1]


def select_tokenizer():
    
    which = models_to_check()
    lms = {w: get_backbone(w) for w in which}
    rows = min(lm.cfg.vocab_size for lm in lms.values())
    cands = discover_tokenizers(rows)
    if not cands:
        raise FileNotFoundError(
            "No usable tokenizer under /kaggle/input: need vocab_50k.json + merges_50k.json (the shared 50k word-BPE) and/or the "
            "byte-level tokenizer.json from the BPE notebook, with <= %d tokens. Attach the tokenizer output as an input." % rows)
    log("tokenizer candidates:", "; ".join(f"{t.kind} ({t.vocab_size} tokens) in {t.where}" for t in cands))
    if len(cands) == 1 and not TOKENIZER_CHECK:
        return cands[0]
    # sample long training documents for the check (deterministic)
    rs = random.Random(SEED)
    pool = [i for i, r in enumerate(raw["train"]) if isinstance(r.get("text"), str) and len(r["text"]) > 8000]
    pick = rs.sample(pool, min(LM_CHECK_DOCS, len(pool))) if pool else []
    texts = [prep_text(raw["train"][i]["text"]) for i in pick]
    if not texts:
        log("WARNING: no document long enough for the tokenizer check - using the first candidate")
        return cands[0]
    refs = {w: (float(P1[w]["val_loss"]) if P1[w]["val_loss"] is not None else 4.5) for w in which}
    scored = []
    for t in cands:
        ids = t.encode_batch(texts)
        losses = {}
        for w in which:
            losses[w], n = lm_next_token_loss(lms[w], ids)
        scored.append((sum(losses.values()), losses, t))
        log(f"  {t.kind:16s} -> " + " | ".join(f"{w} (Model {3 if w == 'teacher' else 4}) next-token loss {losses[w]:.3f} "
                                                f"[pretraining val_loss {refs[w]:.3f}]" for w in which) + f" | {n:,} tokens")
    scored.sort(key=lambda x: (not np.isfinite(x[0]), x[0]))
    _, best_losses, best = scored[0]
    bad = [w for w in which if not np.isfinite(best_losses[w]) or best_losses[w] > refs[w] + LM_CHECK_MARGIN]
    if bad:
        msg = (f"best tokenizer ({best.kind}) gives " + ", ".join(f"{w} loss {best_losses[w]:.2f} vs val_loss {refs[w]:.2f}" for w in bad) +
               ": the tokenizer files do not seem to be the ones these checkpoints were trained with (fine-tuning would start from "
               "noise). Attach the right tokenizer output, or set TOKENIZER_STRICT=False / TOKENIZER_CHECK=False to override.")
        if TOKENIZER_STRICT:
            raise RuntimeError(msg)
        log("WARNING:", msg)
    if len(scored) > 1:
        log(f"  -> using {best.kind}; the other candidate(s) are ignored")
    return best


def _rows_from_config():
    return min(int(P1[w]["cfg"]["vocab_size"]) for w in models_to_check())


tok = select_tokenizer() if TOKENIZER_CHECK else (discover_tokenizers(_rows_from_config()) or [None])[0]
assert tok is not None, "no tokenizer found"
assert tok.vocab_size <= 65535, "token ids must fit uint16"
PAD_ID = tok.pad_id
log(f"TOKENIZER IN USE: {tok.kind} | vocab={tok.vocab_size} pad={tok.pad_id} unk={tok.unk_id} eos={tok.eos_id} | "
    f"fingerprint {tok.fingerprint} | {tok.where}")


# ---- tokenisation of the three splits (cached) ----
def cap_tokens(ids):
    
    if len(ids) <= MAX_DOC_TOKENS:
        return ids, False
    h = int(MAX_DOC_TOKENS * KEEP_HEAD_FRAC)
    return ids[:h] + ids[len(ids) - (MAX_DOC_TOKENS - h):], True


class TokStore:
    """all documents of one split as one flat uint16 array + offsets"""

    def __init__(self, tokens, offsets):
        self.tokens = tokens
        self.offsets = offsets
        self.lens = np.diff(offsets)

    def __len__(self):
        return len(self.lens)


def tokenize_split(name, recs):
    sig = {"v": 3, "n": len(recs), "tok": tok.kind, "tok_fp": tok.fingerprint, "vocab": tok.vocab_size,
           "simplify": SIMPLIFY_PLACEHOLDERS, "max_tokens": MAX_DOC_TOKENS, "head_frac": KEEP_HEAD_FRAC,
           "first": str(recs[0]["id"]), "last": str(recs[-1]["id"])}
    npz, sigp = f"{CACHE_DIR}/{name}_tokens.npz", f"{CACHE_DIR}/{name}_sig.json"
    if os.path.exists(npz) and read_json(sigp) == sig:
        z = np.load(npz)
        log(f"{name}: tokens loaded from cache")
        return TokStore(z["tokens"], z["offsets"])
    t0 = time.time()
    chunks, lens, n_capped = [], [], 0
    BS = 64
    for i in range(0, len(recs), BS):
        texts = [prep_text(r["text"]) for r in recs[i:i + BS]]
        for ids in tok.encode_batch(texts):
            if len(ids) == 0:
                ids = [tok.eos_id]
            ids, capped = cap_tokens(ids)
            n_capped += int(capped)
            chunks.append(np.asarray(ids, dtype=np.uint16))
            lens.append(len(ids))
        if (i // BS) % 40 == 0:
            log(f"  {name}: {i + len(texts):,}/{len(recs):,} docs | {(time.time() - t0) / 60:.1f} min")
    offsets = np.zeros(len(lens) + 1, dtype=np.int64)
    offsets[1:] = np.cumsum(lens)
    tokens = np.concatenate(chunks)
    np.savez(npz, tokens=tokens, offsets=offsets)
    atomic_json(sig, sigp)
    log(f"  {name}: {n_capped} docs hit the {MAX_DOC_TOKENS:,}-token safety cap")
    return TokStore(tokens, offsets)


stores = {s: tokenize_split(s, raw[s]) for s in ("train", "dev", "test")}
for s, st in stores.items():
    q = np.percentile(st.lens, [50, 90, 95, 99])
    log(f"{s:5s}: {len(st):,} docs | tokens/doc median {q[0]:.0f} p90 {q[1]:.0f} p95 {q[2]:.0f} p99 {q[3]:.0f} "
        f"max {st.lens.max():,} | total {st.lens.sum() / 1e6:.1f}M tokens")
_unk = sum(int((st.tokens == tok.unk_id).sum()) for st in stores.values()) / max(sum(len(st.tokens) for st in stores.values()), 1)
log(f"<unk> rate over all splits: {100 * _unk:.3f}%")

TEST_IDS = [r["id"] for r in raw["test"]]          # original ids, original order
for s in raw:                                      # raw texts are no longer needed
    for r in raw[s]:
        r.pop("text", None)
gc.collect()


teacher backbone (Model 3): 8 layers, d_model 640, ffn 1536, 10q/2kv heads, 3 MTP heads, vocab rows 50048
self-test passed: block-local attention == masked reference (max diff 0.00e+00)
student backbone (Model 4): 9 layers ['local_small', 'local_large', 'memory', 'local_small', 'local_large', 'memory', 'local_small', 'local_large', 'memory'], d_model 512, ffn 1792, 8q/2kv heads, 8x4 memory tokens, vocab rows 50000
tokenizer candidates: custom-word-BPE (50000 tokens) in /kaggle/input/datasets/kalgudekrish/bpe-50k
  custom-word-BPE  -> teacher (Model 3) next-token loss 3.469 [pretraining val_loss 2.353] | student (Model 4) next-token loss 3.568 [pretraining val_loss 2.376] | 16,384 tokens
TOKENIZER IN USE: custom-word-BPE | vocab=50000 pad=0 unk=1 eos=2 | fingerprint a69da8d9a861 | /kaggle/input/datasets/kalgudekrish/bpe-50k
  train: 64/42,750 docs | 0.0 min
  train: 2,624/42,750 docs | 0.1 min
  train: 5,184/42,750 docs | 0.1 min
  train: 7,744/42,750 docs | 0.2 min
  train: 10,304/42,7

33

In [5]:
# CELL 5: CHUNK GEOMETRY (all windows), METRICS, THRESHOLD TUNING  (numpy only)


def chunk_starts(n):
    
    return list(range(0, max(1, n - OVERLAP), STRIDE))


def n_chunks_of(n):
    return -(-max(1, n - OVERLAP) // STRIDE)


def encoded_tokens(n):
    """tokens actually pushed through the backbone for a doc of n tokens (overlap counted twice)"""
    k = n_chunks_of(n)
    return n if k == 1 else (k - 1) * CHUNK_LEN + (n - (k - 1) * STRIDE)


# ---- metrics ----------------------------------------------------------------------
def f1_per_label(Y, pred):
    Y, pred = Y.astype(bool), pred.astype(bool)
    tp = (pred & Y).sum(0).astype(np.float64)
    fp = (pred & ~Y).sum(0).astype(np.float64)
    fn = (~pred & Y).sum(0).astype(np.float64)
    den = 2 * tp + fp + fn
    return np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)


def all_metrics(Y, pred):
    Y, pred = Y.astype(bool), pred.astype(bool)
    tp, fp, fn = (pred & Y).sum(), (pred & ~Y).sum(), (~pred & Y).sum()
    return {"macro_f1": float(f1_per_label(Y, pred).mean()),
            "micro_f1": float(2 * tp / max(2 * tp + fp + fn, 1)),
            "subset_acc": float((pred == Y).all(1).mean()),
            "avg_pred_labels": float(pred.sum(1).mean())}


# ---- thresholds -------------------------------------------------------------------
GRID = np.round(np.arange(0.02, 0.96, 0.01), 2)


def f1_curves(P, Y):
    """(len(GRID), L) matrix: F1 of every label at every global threshold"""
    Yb = Y.astype(bool)
    out = np.zeros((len(GRID), P.shape[1]))
    for gi, t in enumerate(GRID):
        out[gi] = f1_per_label(Yb, P >= t)
    return out


def _smooth(curves, w=5):
    pad = w // 2
    c = np.pad(curves, ((pad, pad), (0, 0)), mode="edge")
    k = np.ones(w) / w
    return np.stack([np.convolve(c[:, j], k, mode="valid") for j in range(c.shape[1])], 1)


def fit_thresholds(P, Y, mode="blend", lam=0.5):
    
    curves = f1_curves(P, Y)
    g = GRID[np.argmax(curves.mean(1))]
    t_l = GRID[np.argmax(_smooth(curves), 0)]
    if mode == "global":
        return np.full(P.shape[1], g)
    if mode == "perlabel":
        return t_l.astype(np.float64)
    return lam * t_l + (1 - lam) * g


def apply_thresholds(P, thr, force_top1=True):
    
    pred = P >= thr[None, :]
    if force_top1:
        empty = np.where(~pred.any(1))[0]
        if len(empty):
            pred[empty, np.argmax(P[empty] / thr[None, :], 1)] = True
    return pred


def crossfit_report(P, Y, modes=("global", "perlabel", "blend"), folds=2, seed=0):
    """honest dev estimate: thresholds fitted on the other half of dev, scored on this half"""
    rng = np.random.RandomState(seed)
    perm = rng.permutation(len(P))
    parts = np.array_split(perm, folds)
    res = {}
    for mode in modes:
        pred = np.zeros(P.shape, dtype=bool)
        for k in range(folds):
            te = parts[k]
            tr = np.concatenate([parts[j] for j in range(folds) if j != k])
            thr = fit_thresholds(P[tr], Y[tr], mode)
            pred[te] = apply_thresholds(P[te], thr)
        res[mode] = all_metrics(Y, pred)
    return res


log("chunk audit (EVERY chunk of EVERY document is encoded - no sampling anywhere):")
for _s, _st in stores.items():
    _n = np.array([n_chunks_of(int(x)) for x in _st.lens])
    assert all((k - 1) * STRIDE + CHUNK_LEN >= x for k, x in zip(_n, _st.lens)), "chunk geometry leaves a gap"
    assert _n.max() <= MAX_POS_EMB, f"{_n.max()} chunks > MAX_POS_EMB={MAX_POS_EMB}"
    _enc = sum(encoded_tokens(int(x)) for x in _st.lens)
    log(f"  {_s:5s}: chunks/doc median {np.median(_n):.0f} p90 {np.percentile(_n, 90):.0f} p99 {np.percentile(_n, 99):.0f} "
        f"max {_n.max()} | total chunks {_n.sum():,} | {_enc / 1e6:.1f}M tokens through the backbone per pass "
        f"({_enc / max(_st.lens.sum(), 1):.2f}x overlap factor)")


chunk audit (EVERY chunk of EVERY document is encoded - no sampling anywhere):
  train: chunks/doc median 1 p90 4 p99 12 max 52 | total chunks 86,851 | 63.4M tokens through the backbone per pass (1.22x overlap factor)
  dev  : chunks/doc median 1 p90 4 p99 13 max 52 | total chunks 22,258 | 16.9M tokens through the backbone per pass (1.22x overlap factor)
  test : chunks/doc median 1 p90 4 p99 13 max 52 | total chunks 27,610 | 20.8M tokens through the backbone per pass (1.22x overlap factor)


In [6]:
# CELL 6: BATCHING (token-budget, length-bucketed, ALL chunks) AND INFERENCE

def doc_cost(n_tokens):
    
    return n_chunks_of(n_tokens) * min(CHUNK_LEN, n_tokens)


def pack(doc_order, store, budget, max_docs):
    
    batches, cur, cost = [], [], 0
    for d in doc_order:
        c = doc_cost(int(store.lens[d]))
        if cur and (cost + c > budget or len(cur) >= max_docs):
            batches.append(cur)
            cur, cost = [], 0
        cur.append(int(d))
        cost += c
    if cur:
        batches.append(cur)
    return batches


def make_train_batches(store, epoch):
    
    rng = np.random.RandomState(SEED + 1000 * epoch)
    order = rng.permutation(len(store))
    POOL = 1536
    batches = []
    for p in range(0, len(order), POOL):
        pool = sorted(order[p:p + POOL], key=lambda d: store.lens[d])
        batches += pack(pool, store, TRAIN_TOKEN_BUDGET, MAX_DOCS_PER_BATCH)
    perm = rng.permutation(len(batches))
    return [batches[i] for i in perm]


def make_eval_batches(store):
    order = np.argsort(store.lens)
    return pack(order, store, EVAL_TOKEN_BUDGET, 128)


def collate(store, doc_ids):
    
    seqs, owner, pos = [], [], []
    for bi, d in enumerate(doc_ids):
        a, b = int(store.offsets[d]), int(store.offsets[d + 1])
        n = b - a
        for p, s in enumerate(chunk_starts(n)):
            seqs.append(store.tokens[a + s: a + min(n, s + CHUNK_LEN)])
            owner.append(bi)
            pos.append(p)
    lens = np.array([len(x) for x in seqs], dtype=np.int64)
    ids = np.full((len(seqs), int(lens.max())), PAD_ID, dtype=np.int64)
    for i, x in enumerate(seqs):
        ids[i, :len(x)] = x
    t = lambda a: torch.from_numpy(a).to(DEVICE, non_blocking=True)
    return (t(ids), t(lens), t(np.asarray(owner, dtype=np.int64)), t(np.asarray(pos, dtype=np.int64)), len(doc_ids))


@torch.no_grad()
def predict_probs(model, store, name="", paths=False):
    
    model.eval()
    P = np.zeros((len(store), N_LABELS), dtype=np.float32)
    PP = np.zeros((len(store), N_LABELS, 3), dtype=np.float32) if paths else None
    batches = make_eval_batches(store)
    t0, done = time.time(), 0
    for i, docs in enumerate(batches):
        ids, lens, owner, pos, nd = collate(store, docs)
        with amp_ctx():
            logits, zs, _ = model(ids, lens, owner, pos, nd)
        P[docs] = torch.sigmoid(logits.float()).cpu().numpy()
        if paths:
            PP[docs] = torch.sigmoid(zs.float()).cpu().numpy()
        done += len(docs)
        if i % 50 == 0:
            log(f"  predict {name}: {done:,}/{len(store):,} docs | {(time.time() - t0) / 60:.1f} min")
    if paths:
        return P, {PATH_NAMES[k]: PP[:, :, k] for k in range(3)}
    return P


In [7]:
# CELL 7: TEACHER CACHE

def chunk_offsets(store):
    
    k = np.array([n_chunks_of(int(n)) for n in store.lens], dtype=np.int64)
    off = np.zeros(len(k) + 1, dtype=np.int64)
    off[1:] = np.cumsum(k)
    return off


CACHE_KEYS = ("z", "zs", "alpha", "s", "done")


def new_teacher_cache(store):
    n, coff = len(store), chunk_offsets(store)
    nc = int(coff[-1])
    return {"z": np.zeros((n, N_LABELS), np.float32), "zs": np.zeros((n, N_LABELS, 3), np.float32),
            "alpha": np.zeros((nc, N_LABELS), np.float32), "s": np.zeros((nc, N_LABELS), np.float32),
            "done": np.zeros(n, dtype=bool), "coff": coff}


def teacher_sig(split, store):
    
    return {"v": 1, "split": split, "n_docs": int(len(store)), "n_chunks": int(chunk_offsets(store)[-1]),
            "first": str(raw[split][0]["id"]), "last": str(raw[split][-1]["id"]),
            "teacher_step": TEACHER_META.get("global_step"), "teacher_f1": TEACHER_META.get("dev_macro_f1"),
            "teacher_bytes": int(os.path.getsize(TEACHER_LSI_PATH)), "tok_fp": str(tok.fingerprint),
            "chunk_len": CHUNK_LEN, "stride": STRIDE, "n_labels": int(N_LABELS), "max_tokens": MAX_DOC_TOKENS}


def cache_paths(d, split):
    return f"{d}/teacher_cache_{split}.npz", f"{d}/teacher_cache_{split}_sig.json"


def save_teacher_cache(split, cache, sig):
    npz, sigp = cache_paths(CACHE_DIR, split)
    atomic_json(sig, sigp)
    tmp = npz + ".tmp"
    with open(tmp, "wb") as f:
        np.savez(f, **{k: cache[k] for k in CACHE_KEYS})
    os.replace(tmp, npz)                                 # the cache file is always complete: either the old or the new one


def load_teacher_cache(split, store, sig):
    
    best = None
    for d in [CACHE_DIR] + walk_find(f"teacher_cache_{split}.npz", [INPUT_ROOT]):
        npz, sigp = cache_paths(d, split)
        if not os.path.exists(npz) or read_json(sigp) != sig:
            continue
        try:
            z = np.load(npz)
            c = {k: z[k] for k in CACHE_KEYS}
        except Exception as e:
            log(f"  teacher cache {npz} unreadable ({type(e).__name__}) - ignored")
            continue
        ref = new_teacher_cache(store)
        if any(c[k].shape != ref[k].shape for k in CACHE_KEYS):
            continue
        c["coff"] = ref["coff"]
        n_done = int(c["done"].sum())
        if best is None or n_done > best[0]:
            best = (n_done, c, d)
    if best is None:
        return None
    log(f"  teacher cache '{split}': {best[0]:,}/{len(store):,} documents already done (from {best[2]})")
    return best[1]


def store_teacher_batch(cache, docs, out):
    
    coff = cache["coff"]
    for k in ("z", "zs", "alpha", "s"):
        if not np.isfinite(out[k][out["valid"]] if k in ("alpha", "s") else out[k]).all():
            raise FloatingPointError(f"the teacher produced non-finite values ({k}) - cannot be used as a target")
    for bi, d in enumerate(docs):
        n = int(coff[d + 1] - coff[d])
        v = out["valid"][bi]
        assert int(v.sum()) == n and bool(v[:n].all()), f"doc {d}: {int(v.sum())} chunks from the teacher, expected {n}"
        cache["alpha"][coff[d]:coff[d + 1]] = out["alpha"][bi, :n]
        cache["s"][coff[d]:coff[d + 1]] = out["s"][bi, :n]
    idx = np.asarray(docs)
    cache["z"][idx] = out["z"]
    cache["zs"][idx] = out["zs"]
    cache["done"][idx] = True


def fill_teacher_cache(split, store, cache, sig, run_batch_fn, deadline):
    
    todo = [b for b in make_eval_batches(store) if not cache["done"][b].all()]
    if not todo:
        return True
    n_todo = sum(len(b) for b in todo)
    log(f"  teacher '{split}': {n_todo:,} documents to encode ({len(todo)} batches)")
    t0 = last_save = time.time()
    done = 0
    for i, docs in enumerate(todo):
        if time.time() > deadline:
            save_teacher_cache(split, cache, sig)
            return False
        out = run_batch_fn(docs, store)
        store_teacher_batch(cache, docs, out)
        done += len(docs)
        if time.time() - last_save > TEACHER_CACHE_SAVE_SEC:
            save_teacher_cache(split, cache, sig)
            last_save = time.time()
        if i % 25 == 0:
            el = time.time() - t0
            log(f"  teacher '{split}': {done:,}/{n_todo:,} docs | {el / 60:.1f} min | ETA {fmt_time(el / done * (n_todo - done))}")
    save_teacher_cache(split, cache, sig)
    return True


def build_teacher_model():
    
    ck = torch.load(TEACHER_LSI_PATH, map_location="cpu", weights_only=False)
    sd = ck["trainable"]
    lora_keys = [k for k in sd if re.match(r"lm\.blocks\.\d+\..*\.A$", k)]
    assert lora_keys, "no LoRA tensors in the teacher checkpoint - is this really the Phase-5 Model-3 output?"
    r = int(sd[lora_keys[0]].shape[0])
    targets = {k.split(".")[-2] for k in lora_keys}
    lm = get_backbone("teacher")
    n_lora = inject_lora(lm, targets, r, TEACHER_LORA_ALPHA, 0.0, False, 0.0)
    lm.heads[0].norm.weight.requires_grad = True              # the one backbone norm the Model-3 run trained
    hp = infer_head_hp({k: v for k, v in sd.items() if not k.startswith("lm.")})
    tmodel = LSIModel(lm, teacher_feats, N_LABELS, np.zeros(N_LABELS), hp).to(DEVICE)
    want = {n for n, p in tmodel.named_parameters() if p.requires_grad}
    if want != set(sd):
        raise ValueError(f"teacher weights do not match the rebuilt teacher: missing {sorted(want - set(sd))[:4]} | "
                         f"unexpected {sorted(set(sd) - want)[:4]}")
    load_trainable(tmodel, sd)
    tmodel.eval()
    for p in tmodel.parameters():
        p.requires_grad = False
    log(f"teacher rebuilt: Model 3 + LoRA(r={r}, alpha={TEACHER_LORA_ALPHA:g}, {n_lora} modules: {sorted(targets)}) + head {hp}")
    del ck, sd
    return tmodel


def make_teacher_runner(tmodel):
    @torch.no_grad()
    def run(docs, store):
        ids, lens, owner, pos, nd = collate(store, docs)
        with amp_ctx():
            z, zs, ex = tmodel(ids, lens, owner, pos, nd)
        f = lambda t: t.float().cpu().numpy()
        return {"z": f(z), "zs": f(zs), "alpha": f(ex["alpha"]), "s": f(ex["s"]), "valid": ex["valid"].cpu().numpy()}
    return run


def sigmoid_np(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))


def global_f1(P, Y):
    thr = fit_thresholds(P, Y, "global")
    return all_metrics(Y, apply_thresholds(P, thr)), float(thr[0])


TEACHER = {}              # split -> cache dict (None when the run is a no-teacher baseline)
TEACHER_DEV_F1 = None

if USE_TEACHER:
    deadline = T_START + TEACHER_MAX_H * 3600
    tmodel = runner = None
    for split in ("dev", "train"):               # dev first: cheap, and proves the teacher was rebuilt correctly before the long pass
        store = stores[split]
        sig = teacher_sig(split, store)
        cache = load_teacher_cache(split, store, sig) or new_teacher_cache(store)
        if not cache["done"].all():
            if tmodel is None:
                tmodel = build_teacher_model()
                runner = make_teacher_runner(tmodel)
            if not fill_teacher_cache(split, store, cache, sig, runner, deadline):
                raise RuntimeError(
                    f"the teacher stage used its {TEACHER_MAX_H} h of the session and is not finished ('{split}': "
                    f"{int(cache['done'].sum()):,}/{len(store):,} documents). The cache is saved: use Save & Commit, attach this "
                    f"notebook's output as an input of the next run and run again - the teacher stage resumes where it stopped.")
        else:
            save_teacher_cache(split, cache, sig)     # make sure this session's output also contains it (carried over from an input)
        TEACHER[split] = cache
        if split == "dev":
            m, g = global_f1(sigmoid_np(cache["z"]), Y_dev)
            TEACHER_DEV_F1 = m["macro_f1"]
            log(f"TEACHER on dev (rebuilt from its checkpoint): macro-F1 {m['macro_f1']:.4f} | micro-F1 {m['micro_f1']:.4f} | "
                f"subset-acc {m['subset_acc']:.4f} | global thr {g:.2f}   (its own training run reported {TEACHER_META['dev_macro_f1']:.4f})")
            if not SMOKE_TEST and abs(m["macro_f1"] - TEACHER_META["dev_macro_f1"]) > TEACHER_CHECK_TOL:
                msg = (f"the rebuilt teacher scores {m['macro_f1']:.4f} on dev but its training run reported "
                       f"{TEACHER_META['dev_macro_f1']:.4f}: the rebuilt model is not the one that was trained (config / tokenizer / "
                       f"LoRA scale mismatch). Distilling it would teach the student noise.")
                if TEACHER_STRICT:
                    raise RuntimeError(msg + " Set TEACHER_STRICT=False to override.")
                log("WARNING:", msg)

    P_tr = sigmoid_np(TEACHER["train"]["z"])
    m_tr, _ = global_f1(P_tr, Y_train)
    pos = Y_train.astype(bool)
    log(f"TEACHER on its own TRAIN documents: macro-F1 {m_tr['macro_f1']:.4f} (dev: {TEACHER_DEV_F1:.4f}) | mean p(true statute) "
        f"{P_tr[pos].mean():.3f} | mean p(absent statute) {P_tr[~pos].mean():.4f}")
    if m_tr["macro_f1"] - TEACHER_DEV_F1 > 0.15:
        log("  note: the teacher is much sharper on train than on dev (it has partly memorised the train documents). Its soft targets there "
            "are close to hard labels; the temperature (KD_TEMP) softens them, and the chunk-level evidence terms are what still adds "
            "information. Raise KD_TEMP / lower KD_ALPHA if the student does not beat the plain-LoRA baseline.")
    del tmodel, runner
    BACKBONES.pop("teacher", None)
    P1.pop("teacher", None)
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    log(f"teacher stage finished and freed | session elapsed {fmt_time(time.time() - T_START)}")
else:
    log("no-teacher baseline run: teacher stage skipped")


teacher rebuilt: Model 3 + LoRA(r=32, alpha=64, 56 modules: ['k_proj', 'o_proj', 'q_proj', 'v_proj', 'w_down', 'w_gate', 'w_up']) + head {'hidden': 512, 'dropout': 0.1, 'layers': 1, 'heads': 8, 'ff': 1024, 'max_pos': 64, 'rel_buckets': 8}
  teacher 'dev': 10,181 documents to encode (317 batches)
  teacher 'dev': 128/10,181 docs | 0.0 min | ETA 0h00m
  teacher 'dev': 3,328/10,181 docs | 0.1 min | ETA 0h00m
  teacher 'dev': 6,071/10,181 docs | 0.4 min | ETA 0h00m
  teacher 'dev': 7,007/10,181 docs | 0.7 min | ETA 0h00m
  teacher 'dev': 7,768/10,181 docs | 1.0 min | ETA 0h00m
  teacher 'dev': 8,293/10,181 docs | 1.4 min | ETA 0h00m
  teacher 'dev': 8,770/10,181 docs | 1.7 min | ETA 0h00m
  teacher 'dev': 9,170/10,181 docs | 2.0 min | ETA 0h00m
  teacher 'dev': 9,479/10,181 docs | 2.4 min | ETA 0h00m
  teacher 'dev': 9,736/10,181 docs | 2.7 min | ETA 0h00m
  teacher 'dev': 9,943/10,181 docs | 3.0 min | ETA 0h00m
  teacher 'dev': 10,084/10,181 docs | 3.3 min | ETA 0h00m
  teacher 'dev': 10,

In [8]:
# CELL 8: STUDENT TRAINING

_pos_cnt = Y_train.sum(0).astype(np.float64)
POS_W = torch.tensor(np.clip(np.sqrt((len(Y_train) - _pos_cnt) / np.maximum(_pos_cnt, 1)), 1.0, POS_WEIGHT_CAP),
                     dtype=torch.float32, device=DEVICE)
log(f"pos_weight range: {POS_W.min().item():.2f} .. {POS_W.max().item():.2f}")

Y_train_t = torch.from_numpy(Y_train.astype(np.float32))


def teacher_batch_np(cache, docs):
    
    coff = cache["coff"]
    ks = [int(coff[d + 1] - coff[d]) for d in docs]
    TA = np.zeros((len(docs), max(ks), N_LABELS), np.float32)
    TS = np.zeros_like(TA)
    for bi, d in enumerate(docs):
        TA[bi, :ks[bi]] = cache["alpha"][coff[d]:coff[d + 1]]
        TS[bi, :ks[bi]] = cache["s"][coff[d]:coff[d + 1]]
    idx = np.asarray(docs)
    assert cache["done"][idx].all(), "teacher target requested for a document the teacher has not processed"
    return {"z": cache["z"][idx], "zs": cache["zs"][idx], "alpha": TA, "s": TS}


def teacher_batch(docs):
    return {k: torch.from_numpy(v).to(DEVICE) for k, v in teacher_batch_np(TEACHER["train"], docs).items()}


# ---------------------------- distillation loss ----------------------------
def soft_bce(z, p, T):
    
    return F.binary_cross_entropy_with_logits(z / T, p, reduction="none").mean(1)


def att_kl(a_t, a_s, valid):
    
    t = a_t.clamp_min(1e-8)
    s = a_s.clamp_min(1e-8)
    return (a_t * (t.log() - s.log()) * valid[:, :, None]).sum(1)


def chunk_kl(s_t, s_s, valid, tau):
    
    vm = valid[:, :, None]
    neg = float("-inf")
    lt = torch.log_softmax((s_t / tau).masked_fill(~vm, neg), dim=1).masked_fill(~vm, 0.0)
    ls = torch.log_softmax((s_s / tau).masked_fill(~vm, neg), dim=1).masked_fill(~vm, 0.0)
    return (lt.exp() * vm * (lt - ls)).sum(1)


def distill_loss(z, zs, ex, y, tgt):
    
    def bce(zz):
        return F.binary_cross_entropy_with_logits(zz, y, pos_weight=POS_W, reduction="none").mean(1)

    hard = bce(z) + AUX_W * sum(bce(zs[:, :, k]) for k in range(3)) / 3.0        # every path must be useful on its own
    comp = {"hard": hard}
    loss = hard
    if tgt is None:
        return loss, comp
    T = KD_TEMP
    if KD_ALPHA > 0:
        pt = torch.sigmoid(tgt["z"] / T)
        kd = soft_bce(z, pt, T)
        for k in range(3):
            ptk = torch.sigmoid(tgt["zs"][:, :, k] / T) if PATH_KD_TARGET == "matched" else pt
            kd = kd + (AUX_W / 3.0) * soft_bce(zs[:, :, k], ptk, T)
        kd = kd * (T * T)
        loss = (1.0 - KD_ALPHA) * hard + KD_ALPHA * kd
        comp["kd"] = kd
    if ATT_KD_W > 0 or CHUNK_KD_W > 0:
        assert tgt["alpha"].shape == ex["alpha"].shape, \
            f"teacher/student chunk layout differs: {tuple(tgt['alpha'].shape)} vs {tuple(ex['alpha'].shape)}"
        w = torch.sigmoid(tgt["z"]).detach()                     # (D, L) teacher presence probability
        den = w.sum(1).clamp_min(1.0)
        if ATT_KD_W > 0:
            akl = (att_kl(tgt["alpha"], ex["alpha"], ex["valid"]) * w).sum(1) / den
            loss = loss + ATT_KD_W * akl
            comp["att"] = akl
        if CHUNK_KD_W > 0:
            ckl = (chunk_kl(tgt["s"], ex["s"], ex["valid"], CHUNK_KD_TEMP) * w).sum(1) / den
            loss = loss + CHUNK_KD_W * ckl
            comp["chunk"] = ckl
    if LOGIT_MSE_W > 0:
        mse = ((z - tgt["z"]) ** 2).mean(1)
        loss = loss + LOGIT_MSE_W * mse
        comp["mse"] = mse
    return loss, comp


def make_optimizer(model):
    groups = {"lora": [], "norm": [], "head_w": [], "head_nd": []}
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if n.startswith("lm.") and ("final_norm" in n or n.endswith("mem_init")):
            groups["norm"].append(p)                 # final RMSNorm scale and the latent memory: no weight decay
        elif n.startswith("lm."):
            groups["lora"].append(p)
        elif p.ndim >= 2:
            groups["head_w"].append(p)
        else:
            groups["head_nd"].append(p)
    spec = [("lora", LORA_LR, WEIGHT_DECAY), ("norm", LORA_LR, 0.0),
            ("head_w", HEAD_LR, WEIGHT_DECAY), ("head_nd", HEAD_LR, 0.0)]
    pg = [{"params": groups[k], "lr": lr, "base_lr": lr, "weight_decay": wd} for k, lr, wd in spec if groups[k]]
    return torch.optim.AdamW(pg, betas=(0.9, 0.99), eps=1e-8, fused=(DEVICE.type == "cuda"))


class EMA:
    

    def __init__(self, model, decay):
        self.names = [n for n, p in model.named_parameters() if p.requires_grad]
        self.params = [p for n, p in model.named_parameters() if p.requires_grad]
        self.shadow = [p.detach().clone().float() for p in self.params]
        self.decay = decay

    @torch.no_grad()
    def update(self, step):
        d = min(self.decay, (1.0 + step) / (10.0 + step))
        torch._foreach_mul_(self.shadow, d)
        torch._foreach_add_(self.shadow, [p.detach().float() for p in self.params], alpha=1.0 - d)

    @contextlib.contextmanager
    def applied(self):
        backup = [p.detach().clone() for p in self.params]
        with torch.no_grad():
            for p, s in zip(self.params, self.shadow):
                p.copy_(s)
        try:
            yield
        finally:
            with torch.no_grad():
                for p, b in zip(self.params, backup):
                    p.copy_(b)

    def state_dict(self):
        return {n: s.detach().cpu().clone() for n, s in zip(self.names, self.shadow)}

    def load_state_dict(self, sd):
        for n, s in zip(self.names, self.shadow):
            s.copy_(sd[n].to(s.device))


def lr_mult(step, prog):
    
    warm = min(1.0, (step + 1) / WARMUP_STEPS)
    return warm * (MIN_LR_RATIO + (1 - MIN_LR_RATIO) * 0.5 * (1 + math.cos(math.pi * min(1.0, prog))))


def find_previous_latest():
    best = None
    for d in walk_find("lsi_latest_meta.json", [CKPT_DIR, INPUT_ROOT]):
        meta = read_json(os.path.join(d, "lsi_latest_meta.json"))
        if not meta or meta.get("tag") != RUN_TAG or meta.get("n_labels") != N_LABELS:
            continue
        if not os.path.exists(os.path.join(d, "lsi_latest.pt")):
            continue
        if best is None or meta["global_step"] > best[1]["global_step"]:
            best = (d, meta)
    return best


def carry_over_best():
    
    cur = read_json(f"{CKPT_DIR}/lsi_best_meta.json")
    for d in walk_find("lsi_best_meta.json", [INPUT_ROOT]):
        meta = read_json(os.path.join(d, "lsi_best_meta.json"))
        if not meta or meta.get("tag") != RUN_TAG or not os.path.exists(os.path.join(d, "lsi_best.pt")):
            continue
        if cur is None or meta["dev_macro_f1"] > cur["dev_macro_f1"]:
            shutil.copy(os.path.join(d, "lsi_best.pt"), f"{CKPT_DIR}/lsi_best.pt.tmp")
            os.replace(f"{CKPT_DIR}/lsi_best.pt.tmp", f"{CKPT_DIR}/lsi_best.pt")
            shutil.copy(os.path.join(d, "lsi_best_meta.json"), f"{CKPT_DIR}/lsi_best_meta.json")
            cur = meta
    return cur


def dev_eval(model):
    
    P = predict_probs(model, stores["dev"], "dev")
    thr = fit_thresholds(P, Y_dev, "global")
    m = all_metrics(Y_dev, apply_thresholds(P, thr))
    ref = f" | teacher {TEACHER_DEV_F1:.4f}" if TEACHER_DEV_F1 is not None else ""
    log(f"  DEV  macro-F1 {m['macro_f1']:.4f}{ref} | micro-F1 {m['micro_f1']:.4f} | subset-acc {m['subset_acc']:.4f} "
        f"| global thr {thr[0]:.2f} | avg labels/doc {m['avg_pred_labels']:.2f}")
    return m


KD_SETTINGS = {"alpha": KD_ALPHA, "temp": KD_TEMP, "att_w": ATT_KD_W, "chunk_w": CHUNK_KD_W, "chunk_temp": CHUNK_KD_TEMP,
               "logit_mse_w": LOGIT_MSE_W, "path_target": PATH_KD_TARGET, "use_teacher": USE_TEACHER,
               "teacher_dev_f1": TEACHER_DEV_F1, "teacher_run": (TEACHER_META or {}).get("tag")}


def train(model):
    best_meta = carry_over_best()
    best_f1 = best_meta["dev_macro_f1"] if best_meta else -1.0
    opt = make_optimizer(model)
    scaler = make_grad_scaler(AMP["enabled"] and AMP["dtype"] == torch.float16)
    state = {"epoch": 0, "bi": 0, "global_step": 0, "train_sec": 0.0}
    ck_ema = None

    prev = find_previous_latest()
    if prev is not None:
        d, meta = prev
        ck = torch.load(os.path.join(d, "lsi_latest.pt"), map_location="cpu", weights_only=False)
        load_trainable(model, ck["trainable"])
        opt.load_state_dict(ck["opt"])
        scaler.load_state_dict(ck["scaler"])
        state.update(epoch=ck["epoch"], bi=ck["bi"], global_step=ck["global_step"], train_sec=ck.get("train_sec", 0.0))
        ck_ema = ck.get("ema")
        log(f"RESUMED from {d}: epoch {state['epoch'] + 1}, batch {state['bi']}, step {state['global_step']}, "
            f"train time so far {fmt_time(state['train_sec'])}")
        del ck
    ema = EMA(model, EMA_DECAY) if USE_EMA else None
    if ema is not None and ck_ema is not None:
        ema.load_state_dict(ck_ema)
    train_params = [p for g in opt.param_groups for p in g["params"]]

    n_batches0 = len(make_train_batches(stores["train"], 0))
    total_steps = EPOCHS * n_batches0
    n_chunks_train = sum(n_chunks_of(int(n)) for n in stores["train"].lens)
    log(f"{len(stores['train']):,} train docs | {n_chunks_train:,} chunks per epoch (ALL used) | ~{n_batches0:,} steps/epoch "
        f"| {EPOCHS} epochs | ~{total_steps:,} steps")
    log("loss: " + (f"(1-{KD_ALPHA}) hard + {KD_ALPHA} x T^2 softBCE(T={KD_TEMP}) + {ATT_KD_W} x attention-KL + {CHUNK_KD_W} x chunk-logit-KL"
                    if USE_TEACHER else "hard BCE only (no teacher)"))


    sess_avail = (TIME_BUDGET_H - FINAL_RESERVE_H) * 3600 - (time.time() - T_START)       # the teacher stage is already deducted
    train_budget = min(TRAIN_HOURS * 3600, state["train_sec"] + max(sess_avail, 0.0))
    log(f"training-time budget: {fmt_time(train_budget)} in total ({fmt_time(state['train_sec'])} already used) | "
        f"session wall-clock limit {TIME_BUDGET_H}h, {FINAL_RESERVE_H}h reserved for final prediction")

    last = [time.time()]

    def tick():
        now = time.time()
        state["train_sec"] += now - last[0]
        last[0] = now

    def progress():
        return min(1.0, max(state["global_step"] / total_steps, state["train_sec"] / max(train_budget, 1.0)))

    def save_latest():
        atomic_torch_save({"trainable": trainable_state(model), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                           "ema": ema.state_dict() if ema is not None else None,
                           "epoch": state["epoch"], "bi": state["bi"], "global_step": state["global_step"],
                           "train_sec": state["train_sec"], "tag": RUN_TAG}, f"{CKPT_DIR}/lsi_latest.pt")
        atomic_json({"tag": RUN_TAG, "n_labels": N_LABELS, "epoch": state["epoch"], "bi": state["bi"],
                     "global_step": state["global_step"], "train_sec": state["train_sec"], "kd": KD_SETTINGS,
                     "saved_at": time.strftime("%Y-%m-%d %H:%M:%S")}, f"{CKPT_DIR}/lsi_latest_meta.json")

    def eval_and_maybe_save_best(tag):
        nonlocal best_f1
        with (ema.applied() if ema is not None else contextlib.nullcontext()):      # dev score of the EMA weights
            m = dev_eval(model)
            if m["macro_f1"] > best_f1:
                best_f1 = m["macro_f1"]
                atomic_torch_save({"trainable": trainable_state(model), "dev_macro_f1": best_f1, "tag": RUN_TAG},
                                  f"{CKPT_DIR}/lsi_best.pt")
                atomic_json({"tag": RUN_TAG, "dev_macro_f1": best_f1, "where": tag, "global_step": state["global_step"],
                             "label_ids": LABEL_IDS, "n_labels": N_LABELS, "kd": KD_SETTINGS}, f"{CKPT_DIR}/lsi_best_meta.json")
                log(f"  >> NEW BEST dev macro-F1 {best_f1:.4f} ({tag}) - lsi_best.pt saved")
        return m

    timed_out = False
    t_loop = time.time()
    steps_this_run, chunks_seen, tokens_seen, n_skipped = 0, 0, 0, 0
    run = {}                                                   # running means of the loss components
    run_n = 0

    for epoch in range(state["epoch"], EPOCHS):
        batches = make_train_batches(stores["train"], epoch)
        model.train()
        for bi in range(state["bi"], len(batches)):
            if state["train_sec"] >= train_budget:
                timed_out = True
                break
            docs = batches[bi]
            ids, lens, owner, pos, nd = collate(stores["train"], docs)       # ALL chunks of every doc in the batch
            y = Y_train_t[docs].to(DEVICE)
            tgt = teacher_batch(docs) if USE_TEACHER else None

            mult = lr_mult(state["global_step"], progress())
            for g in opt.param_groups:
                g["lr"] = g["base_lr"] * mult

            with amp_ctx():
                logits, zs, ex = model(ids, lens, owner, pos, nd)
            per_doc, comp = distill_loss(logits.float(), zs.float(), ex, y, tgt)
            loss = per_doc.sum() / DOC_NORM
            opt.zero_grad(set_to_none=True)
            if torch.isfinite(loss):
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(train_params, GRAD_CLIP)
                scaler.step(opt)
                scaler.update()
                if ema is not None:
                    ema.update(state["global_step"])
                for k, v in comp.items():
                    run[k] = run.get(k, 0.0) + v.mean().item()
                run_n += 1
            else:
                n_skipped += 1
                log(f"  WARNING: non-finite loss at step {state['global_step']} (batch of {len(docs)} docs) - step skipped")

            state["global_step"] += 1
            state["epoch"], state["bi"] = epoch, bi + 1
            steps_this_run += 1
            chunks_seen += int(lens.numel())
            tokens_seen += int(lens.sum().item())
            tick()

            if steps_this_run % LOG_EVERY == 0:
                el = time.time() - t_loop
                eta_epoch = (len(batches) - bi - 1) * el / steps_this_run
                comps = " ".join(f"{k} {v / max(run_n, 1):.4f}" for k, v in run.items())
                log(f"ep {epoch + 1}/{EPOCHS} step {bi + 1}/{len(batches)} (global {state['global_step']}) | {comps} | "
                    f"lr x{mult:.3f} | {chunks_seen / el:.1f} chunks/s {tokens_seen / el / 1000:.1f}k tok/s | "
                    f"epoch ETA {fmt_time(eta_epoch)} | schedule {100 * progress():.0f}% | "
                    f"train time {fmt_time(state['train_sec'])}/{fmt_time(train_budget)}")
                run, run_n = {}, 0
            if state["global_step"] % SAVE_EVERY == 0:
                save_latest()
                tick()

        if timed_out:
            log(f"training-time budget reached inside epoch {epoch + 1} - saving, then dev tuning / prediction")
            save_latest()
            eval_and_maybe_save_best(f"timeout@epoch{epoch + 1}")
            tick()
            break
        state["epoch"], state["bi"] = epoch + 1, 0
        save_latest()
        eval_and_maybe_save_best(f"epoch{epoch + 1}")
        tick()
        model.train()

    if n_skipped:
        log(f"note: {n_skipped} step(s) skipped because of a non-finite loss")
    log(f"training finished | best dev macro-F1 (global thr) {best_f1:.4f} | train time {fmt_time(state['train_sec'])} "
        f"| session elapsed {fmt_time(time.time() - T_START)}")
    return best_f1


model = build_student()
BEST_DEV_F1 = train(model)


pos_weight range: 1.63 .. 5.00
student LoRA modules: 78 | trainable 6.31M / total 66.7M params (9.5%)
42,750 train docs | 86,851 chunks per epoch (ALL used) | ~3,236 steps/epoch | 5 epochs | ~16,180 steps
loss: (1-0.5) hard + 0.5 x T^2 softBCE(T=2.0) + 0.3 x attention-KL + 0.3 x chunk-logit-KL
training-time budget: 10h39m in total (0h00m already used) | session wall-clock limit 12.0h, 1.0h reserved for final prediction
ep 1/5 step 50/3236 (global 50) | hard 0.5700 kd 2.0630 att 0.0251 chunk 0.4873 | lr x0.250 | 23.9 chunks/s 17.8k tok/s | epoch ETA 0h55m | schedule 0% | train time 0h00m/10h39m
ep 1/5 step 100/3236 (global 100) | hard 0.4922 kd 1.9276 att 0.0237 chunk 0.4106 | lr x0.500 | 24.5 chunks/s 18.2k tok/s | epoch ETA 0h54m | schedule 1% | train time 0h01m/10h39m
ep 1/5 step 150/3236 (global 150) | hard 0.4024 kd 1.8573 att 0.0175 chunk 0.2985 | lr x0.750 | 25.5 chunks/s 18.2k tok/s | epoch ETA 0h53m | schedule 1% | train time 0h02m/10h39m
ep 1/5 step 200/3236 (global 200) | har

In [9]:
# CELL 9: EVALUATION AND SUBMISSION

# (every chunk of every dev / test document is scored)

bm = read_json(f"{CKPT_DIR}/lsi_best_meta.json")
assert bm is not None and os.path.exists(f"{CKPT_DIR}/lsi_best.pt"), "no best checkpoint found - did training run?"
assert bm["label_ids"] == LABEL_IDS, "label space differs from the one the checkpoint was trained with"
ck = torch.load(f"{CKPT_DIR}/lsi_best.pt", map_location="cpu", weights_only=False)
load_trainable(model, ck["trainable"])
log(f"loaded best checkpoint (dev macro-F1 {ck['dev_macro_f1']:.4f}, {bm['where']})")

# ---- 1) dev probabilities (fused + the 3 aggregation paths) ----
P_dev, P_dev_paths = predict_probs(model, stores["dev"], "dev", paths=True)

log("\nAggregation-path ablation on dev (global threshold, >=1 label/doc; paths were trained jointly, so this shows what each adds):")
for name, Pp in list(P_dev_paths.items()) + [("FUSED (used)", P_dev)]:
    thr_g = fit_thresholds(Pp, Y_dev, "global")
    m = all_metrics(Y_dev, apply_thresholds(Pp, thr_g))
    log(f"  {name:13s} macro-F1 {m['macro_f1']:.4f} | micro-F1 {m['micro_f1']:.4f} | subset-acc {m['subset_acc']:.4f}")

cf = crossfit_report(P_dev, Y_dev)          # thresholds fitted on one half of dev, scored on the other half
log("\nDEV, honest cross-fitted estimate of the fused model (force >=1 label per document):")
for mode, m in cf.items():
    log(f"  {mode:9s} macro-F1 {m['macro_f1']:.4f} | micro-F1 {m['micro_f1']:.4f} | "
        f"subset-acc {m['subset_acc']:.4f} | avg labels/doc {m['avg_pred_labels']:.2f}")
BEST_MODE = max(cf, key=lambda k: cf[k]["macro_f1"])
THR = fit_thresholds(P_dev, Y_dev, BEST_MODE)          # final thresholds use all of dev
pred_dev = apply_thresholds(P_dev, THR)
m_dev = all_metrics(Y_dev, pred_dev)
log(f"chosen strategy: {BEST_MODE} | in-sample dev macro-F1 {m_dev['macro_f1']:.4f} (optimistic, as thresholds were fitted on it)")
log(f"thresholds: min {THR.min():.2f} | median {np.median(THR):.2f} | max {THR.max():.2f}")


TEACHER_REPORT = None
if USE_TEACHER:
    P_dev_t = sigmoid_np(TEACHER["dev"]["z"])
    cf_t = crossfit_report(P_dev_t, Y_dev)
    log("\nSTUDENT (Model 4 + LoRA, distilled)  vs  TEACHER (Model 3 + LoRA), dev, cross-fitted thresholds:")
    for mode in cf:
        log(f"  {mode:9s} macro-F1 {cf[mode]['macro_f1']:.4f} vs {cf_t[mode]['macro_f1']:.4f} | micro-F1 {cf[mode]['micro_f1']:.4f} vs "
            f"{cf_t[mode]['micro_f1']:.4f} | subset-acc {cf[mode]['subset_acc']:.4f} vs {cf_t[mode]['subset_acc']:.4f}")
    blend = {}
    for w in (0.25, 0.5, 0.75):
        blend[str(w)] = crossfit_report(w * P_dev + (1 - w) * P_dev_t, Y_dev, modes=(BEST_MODE,))[BEST_MODE]
        log(f"  info only: {w:.2f} x student + {1 - w:.2f} x teacher ({BEST_MODE}) macro-F1 {blend[str(w)]['macro_f1']:.4f} "
            f"(the submission uses the student alone)")
    TEACHER_REPORT = {"teacher_dev_crossfit": cf_t, "student_dev_crossfit": cf, "blend_dev_crossfit": blend}

f1_l = f1_per_label(Y_dev, pred_dev)
worst = np.argsort(f1_l)[:8]
log("weakest statutes on dev (id: F1, train positives): " +
    ", ".join(f"{LABEL_IDS[i]}: {f1_l[i]:.2f} ({int(Y_train[:, i].sum())})" for i in worst))

# ---- 2) test prediction ----
P_test = predict_probs(model, stores["test"], "test")
pred_test = apply_thresholds(P_test, THR)

preds = []
valid_ids = set(LABEL_IDS)
for i, doc_id in enumerate(TEST_IDS):
    ids = sorted({int(LABEL_IDS[j]) for j in np.where(pred_test[i])[0]})       # unique ints, valid statute ids only
    assert ids and all(x in valid_ids for x in ids)
    preds.append({"id": doc_id, "pred": ids})

assert len(preds) == len(TEST_IDS) == len(stores["test"])
assert len({str(p["id"]) for p in preds}) == len(preds), "duplicate test ids"
log(f"test: {len(preds):,} predictions | avg statutes/doc {np.mean([len(p['pred']) for p in preds]):.2f} "
    f"(dev truth: {Y_dev.sum(1).mean():.2f})")

# ---- 3) save everything ----
pred_path = f"{OUT_DIR}/prediction.json"
with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(preds, f, ensure_ascii=False)
zip_path = f"{OUT_DIR}/submission.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(pred_path, arcname="prediction.json")            # prediction.json at the zip root

np.save(f"{OUT_DIR}/dev_probs.npy", P_dev)                   # keep for ensembling with the other Phase-1 models
np.save(f"{OUT_DIR}/test_probs.npy", P_test)
atomic_json({"label_ids": LABEL_IDS, "thresholds": [float(x) for x in THR], "strategy": BEST_MODE,
             "dev_crossfit": cf, "dev_in_sample": m_dev, "test_ids": [str(x) for x in TEST_IDS],
             "tokenizer": {"kind": tok.kind, "fingerprint": tok.fingerprint, "where": tok.where}, "run_tag": RUN_TAG,
             "kd": KD_SETTINGS, "student_vs_teacher": TEACHER_REPORT},
            f"{OUT_DIR}/thresholds_and_report.json")
log("saved:", pred_path, "|", zip_path, "| dev_probs.npy | test_probs.npy | thresholds_and_report.json")
log("done.")


loaded best checkpoint (dev macro-F1 0.4200, epoch5)
  predict dev: 128/10,181 docs | 0.0 min
  predict dev: 6,071/10,181 docs | 0.5 min
  predict dev: 7,768/10,181 docs | 1.2 min
  predict dev: 8,770/10,181 docs | 1.9 min
  predict dev: 9,479/10,181 docs | 2.6 min
  predict dev: 9,943/10,181 docs | 3.3 min
  predict dev: 10,160/10,181 docs | 3.9 min

Aggregation-path ablation on dev (global threshold, >=1 label/doc; paths were trained jointly, so this shows what each adds):
  attention     macro-F1 0.4190 | micro-F1 0.5244 | subset-acc 0.0632
  chunk-LSE     macro-F1 0.4183 | micro-F1 0.5241 | subset-acc 0.0560
  mean          macro-F1 0.4181 | micro-F1 0.5211 | subset-acc 0.0610
  FUSED (used)  macro-F1 0.4200 | micro-F1 0.5255 | subset-acc 0.0600

DEV, honest cross-fitted estimate of the fused model (force >=1 label per document):
  global    macro-F1 0.4193 | micro-F1 0.5261 | subset-acc 0.0621 | avg labels/doc 4.51
  perlabel  macro-F1 0.4188 | micro-F1 0.5201 | subset-acc 0.0582 